# Saturday Ops, Day 1: Build the Semantic Layer
**AI Bootcamp | Semantic Views, Cortex Search and Cortex AI Functions**

You run data for the **UCLA athletic department**. Game day is a business: tickets, concessions, merch, donors and fan experience. Football Ops also wants to understand how every team in the conference really plays. Today you will:

1. **Generate the data** with a single SQL cell (structured tables plus unstructured text).
2. **Use Cortex AI Functions** to turn fan surveys, service tickets, game recaps and coach press conferences into structured insights.
3. **Build a Semantic View** so business users can ask questions in plain English through Cortex Analyst.
4. **Create a Cortex Search service** over coach press conferences.
5. **Merge structured and unstructured data** to catch a coach who says one thing while the stats say another.

> **Disclaimer:** All data in this notebook is synthetic and generated for training purposes. School names are used for illustration only. Coaches and players are generic aliases (for example "BYU Head Coach", QB1, RB1). Scores, stats, quotes and business figures are fictional and do not describe any real program or person.

**Prerequisites:** a Snowflake account with Cortex AI enabled and a role that can create a database and a warehouse (ACCOUNTADMIN or SYSADMIN in a trial account works). Run the cells top to bottom. The full notebook runs in about 5 minutes on an XS warehouse.

## 0. Setup
Creates the `SATURDAY_OPS` database, two schemas and an XS warehouse. Change `ROLE` if you are not using ACCOUNTADMIN.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

ROLE = "ACCOUNTADMIN"  # change if you use a different role

for stmt in [
    f"USE ROLE {ROLE}",
    "CREATE DATABASE IF NOT EXISTS SATURDAY_OPS",
    "CREATE SCHEMA IF NOT EXISTS SATURDAY_OPS.RAW",
    "CREATE SCHEMA IF NOT EXISTS SATURDAY_OPS.ANALYTICS",
    "CREATE WAREHOUSE IF NOT EXISTS SATURDAY_OPS_WH WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE",
    "USE WAREHOUSE SATURDAY_OPS_WH",
    "USE SCHEMA SATURDAY_OPS.ANALYTICS",
]:
    session.sql(stmt).collect()

print("Ready:", session.sql("SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE()").collect()[0])

## 1. Generate the data (one cell)
This single SQL cell builds every table you need. No files, stages or integrations. It runs in under a minute and always produces identical data.

| Table | Rows | What it is |
|---|---|---|
| TEAMS, PLAYERS, GAMES, TEAM_GAME_STATS | 12, 180, 66, 132 | A 12-team, 11-week conference season |
| FANS, TICKET_SALES, CONCESSION_SALES, MERCH_SALES, DONATIONS | 50K, ~480K, 1M, 120K, 15K | UCLA home game business at the Rose Bowl |
| GAME_RECAPS, PRESSER_TRANSCRIPTS | 132 each | Unstructured: media recaps and coach press conferences |
| FAN_SURVEYS, FAN_SERVICE_TICKETS | 4K, 2K | Unstructured: fan voice |

The text is written to match each game's real stats, with one deliberate twist hidden inside. Can you find it by the end?

In [ ]:
%%sql -r generate_data
EXECUTE IMMEDIATE $$
BEGIN
  -- Deterministic pseudo-random helper: same inputs always produce the same 0-1 value,
  -- so every attendee gets identical data.
  CREATE OR REPLACE FUNCTION SATURDAY_OPS.RAW.RND(ID NUMBER, K VARCHAR)
    RETURNS FLOAT AS '((ABS(HASH(ID, K)) % 1000003) / 1000003.0)::FLOAT';

  ---------------------------------------------------------------------------
  -- Reference data: teams, players, games, team game stats
  ---------------------------------------------------------------------------
  CREATE OR REPLACE TABLE SATURDAY_OPS.RAW.TEAMS (
    TEAM_ID INT, SCHOOL_NAME STRING, MASCOT STRING, CITY STRING, STATE STRING,
    STADIUM STRING, TRUE_STYLE STRING, COACH_ALIAS STRING);
  INSERT INTO SATURDAY_OPS.RAW.TEAMS
    SELECT $1, $2, $3, $4, $5, $6, $7, $2 || ' Head Coach' FROM VALUES
    (1, 'UCLA', 'Bruins', 'Pasadena', 'CA', 'Rose Bowl', 'BALANCED'),
        (2, 'USC', 'Trojans', 'Los Angeles', 'CA', 'Los Angeles Memorial Coliseum', 'TEMPO_SPREAD'),
        (3, 'Texas Tech', 'Red Raiders', 'Lubbock', 'TX', 'Jones AT&T Stadium', 'AIR_RAID'),
        (4, 'Ole Miss', 'Rebels', 'Oxford', 'MS', 'Vaught-Hemingway Stadium', 'TEMPO_SPREAD'),
        (5, 'Iowa', 'Hawkeyes', 'Iowa City', 'IA', 'Kinnick Stadium', 'DEFENSIVE_GRIND'),
        (6, 'Utah', 'Utes', 'Salt Lake City', 'UT', 'Rice-Eccles Stadium', 'GROUND_AND_POUND'),
        (7, 'Georgia', 'Bulldogs', 'Athens', 'GA', 'Sanford Stadium', 'BALANCED'),
        (8, 'BYU', 'Cougars', 'Provo', 'UT', 'LaVell Edwards Stadium', 'GROUND_AND_POUND'),
        (9, 'Ohio State', 'Buckeyes', 'Columbus', 'OH', 'Ohio Stadium', 'AIR_RAID'),
        (10, 'Notre Dame', 'Fighting Irish', 'Notre Dame', 'IN', 'Notre Dame Stadium', 'GROUND_AND_POUND'),
        (11, 'Miami', 'Hurricanes', 'Miami Gardens', 'FL', 'Hard Rock Stadium', 'AIR_RAID'),
        (12, 'Indiana', 'Hoosiers', 'Bloomington', 'IN', 'Memorial Stadium', 'DEFENSIVE_GRIND');

  CREATE OR REPLACE TABLE SATURDAY_OPS.RAW.PLAYERS AS
    SELECT ROW_NUMBER() OVER (ORDER BY t.TEAM_ID, p.SEQ) AS PLAYER_ID, t.TEAM_ID,
           p.ALIAS, p.POSITION
    FROM SATURDAY_OPS.RAW.TEAMS t
    CROSS JOIN (SELECT * FROM VALUES
      (1,'QB1','QB'),(2,'QB2','QB'),(3,'RB1','RB'),(4,'RB2','RB'),(5,'WR1','WR'),
      (6,'WR2','WR'),(7,'WR3','WR'),(8,'TE1','TE'),(9,'OL1','OL'),(10,'DE1','DL'),
      (11,'DT1','DL'),(12,'LB1','LB'),(13,'LB2','LB'),(14,'CB1','DB'),(15,'S1','DB')
      AS v(SEQ, ALIAS, POSITION)) p;

  CREATE OR REPLACE TABLE SATURDAY_OPS.RAW.GAMES (
    GAME_ID INT, SEASON_WEEK INT, GAME_DATE DATE, KICKOFF_TIME STRING, KICKOFF_WINDOW STRING,
    HOME_TEAM_ID INT, AWAY_TEAM_ID INT, HOME_SCORE INT, AWAY_SCORE INT, IS_RIVALRY BOOLEAN,
    HOME_RANK INT, AWAY_RANK INT, WEATHER STRING, TEMP_F INT);
  INSERT INTO SATURDAY_OPS.RAW.GAMES SELECT * FROM VALUES
    (1, 1, '2025-09-06', '12:00', 'NOON', 1, 12, 24, 17, FALSE, 14, NULL, 'HEAT', 98),
        (2, 1, '2025-09-06', '12:00', 'NOON', 7, 2, 39, 33, FALSE, 3, 22, 'CLEAR', 66),
        (3, 1, '2025-09-06', '15:30', 'AFTERNOON', 8, 6, 31, 34, FALSE, 12, NULL, 'HEAT', 98),
        (4, 1, '2025-09-06', '12:00', 'NOON', 5, 10, 32, 14, FALSE, NULL, NULL, 'CLEAR', 78),
        (5, 1, '2025-09-06', '15:30', 'AFTERNOON', 3, 9, 54, 27, FALSE, 18, 5, 'HEAT', 94),
        (6, 1, '2025-09-06', '12:00', 'NOON', 4, 11, 31, 29, FALSE, 9, NULL, 'CLEAR', 81),
        (7, 2, '2025-09-13', '19:30', 'NIGHT', 1, 7, 27, 31, FALSE, 14, 3, 'CLEAR', 72),
        (8, 2, '2025-09-13', '12:00', 'NOON', 6, 12, 7, 19, FALSE, NULL, NULL, 'HEAT', 100),
        (9, 2, '2025-09-13', '12:00', 'NOON', 10, 2, 23, 36, FALSE, NULL, 22, 'CLEAR', 68),
        (10, 2, '2025-09-13', '15:30', 'AFTERNOON', 8, 3, 19, 32, FALSE, 12, 18, 'CLEAR', 75),
        (11, 2, '2025-09-13', '15:30', 'AFTERNOON', 4, 5, 21, 18, FALSE, 9, NULL, 'CLEAR', 78),
        (12, 2, '2025-09-13', '12:00', 'NOON', 9, 11, 41, 42, FALSE, 5, NULL, 'CLEAR', 79),
        (13, 3, '2025-09-20', '15:30', 'AFTERNOON', 6, 1, 17, 20, FALSE, NULL, 14, 'CLEAR', 64),
        (14, 3, '2025-09-20', '19:30', 'NIGHT', 10, 7, 25, 26, FALSE, NULL, 3, 'CLEAR', 81),
        (15, 3, '2025-09-20', '15:30', 'AFTERNOON', 3, 12, 28, 18, FALSE, 18, NULL, 'CLEAR', 71),
        (16, 3, '2025-09-20', '15:30', 'AFTERNOON', 2, 4, 37, 27, FALSE, 22, 9, 'HEAT', 95),
        (17, 3, '2025-09-20', '15:30', 'AFTERNOON', 11, 8, 30, 23, FALSE, NULL, 12, 'CLEAR', 73),
        (18, 3, '2025-09-20', '15:30', 'AFTERNOON', 5, 9, 13, 36, FALSE, NULL, 5, 'CLEAR', 75),
        (19, 4, '2025-09-27', '15:30', 'AFTERNOON', 1, 10, 13, 10, FALSE, 14, NULL, 'RAIN', 58),
        (20, 4, '2025-09-27', '15:30', 'AFTERNOON', 3, 6, 28, 25, FALSE, 18, NULL, 'CLEAR', 60),
        (21, 4, '2025-09-27', '15:30', 'AFTERNOON', 7, 4, 27, 25, FALSE, 3, 9, 'CLEAR', 74),
        (22, 4, '2025-09-27', '15:30', 'AFTERNOON', 11, 12, 31, 19, FALSE, NULL, NULL, 'CLEAR', 73),
        (23, 4, '2025-09-27', '15:30', 'AFTERNOON', 2, 9, 30, 38, FALSE, 22, 5, 'CLEAR', 74),
        (24, 4, '2025-09-27', '19:30', 'NIGHT', 5, 8, 23, 19, FALSE, NULL, 12, 'HEAT', 92),
        (25, 5, '2025-10-04', '19:30', 'NIGHT', 3, 1, 41, 28, FALSE, 18, 14, 'CLEAR', 70),
        (26, 5, '2025-10-04', '15:30', 'AFTERNOON', 10, 4, 26, 24, FALSE, NULL, 9, 'CLEAR', 66),
        (27, 5, '2025-10-04', '12:00', 'NOON', 6, 11, 32, 21, FALSE, NULL, NULL, 'CLEAR', 66),
        (28, 5, '2025-10-04', '15:30', 'AFTERNOON', 9, 7, 33, 35, FALSE, 5, 3, 'CLEAR', 66),
        (29, 5, '2025-10-04', '15:30', 'AFTERNOON', 5, 12, 13, 16, FALSE, NULL, NULL, 'CLEAR', 68),
        (30, 5, '2025-10-04', '12:00', 'NOON', 8, 2, 34, 25, FALSE, 12, 22, 'CLEAR', 65),
        (31, 6, '2025-10-11', '15:30', 'AFTERNOON', 4, 1, 35, 38, FALSE, 9, 14, 'CLEAR', 78),
        (32, 6, '2025-10-11', '15:30', 'AFTERNOON', 11, 3, 24, 34, FALSE, NULL, 18, 'CLEAR', 82),
        (33, 6, '2025-10-11', '12:00', 'NOON', 10, 9, 27, 35, FALSE, NULL, 5, 'CLEAR', 82),
        (34, 6, '2025-10-11', '15:30', 'AFTERNOON', 5, 6, 17, 29, FALSE, NULL, NULL, 'CLEAR', 73),
        (35, 6, '2025-10-11', '12:00', 'NOON', 8, 7, 31, 26, FALSE, 12, 3, 'CLEAR', 76),
        (36, 6, '2025-10-11', '15:30', 'AFTERNOON', 12, 2, 37, 10, FALSE, NULL, 22, 'HEAT', 96),
        (37, 7, '2025-10-18', '15:30', 'AFTERNOON', 1, 11, 17, 34, FALSE, 14, NULL, 'CLEAR', 80),
        (38, 7, '2025-10-18', '15:30', 'AFTERNOON', 4, 9, 22, 28, FALSE, 9, 5, 'CLEAR', 81),
        (39, 7, '2025-10-18', '15:30', 'AFTERNOON', 5, 3, 28, 34, FALSE, NULL, 18, 'CLEAR', 68),
        (40, 7, '2025-10-18', '12:00', 'NOON', 10, 8, 16, 31, FALSE, NULL, 12, 'CLEAR', 72),
        (41, 7, '2025-10-18', '12:00', 'NOON', 2, 6, 24, 27, FALSE, 22, NULL, 'CLEAR', 61),
        (42, 7, '2025-10-18', '15:30', 'AFTERNOON', 12, 7, 26, 22, FALSE, NULL, 3, 'HEAT', 100),
        (43, 8, '2025-10-25', '12:00', 'NOON', 9, 1, 38, 21, FALSE, 5, 14, 'COLD', 44),
        (44, 8, '2025-10-25', '19:30', 'NIGHT', 5, 11, 17, 27, FALSE, NULL, NULL, 'CLEAR', 61),
        (45, 8, '2025-10-25', '15:30', 'AFTERNOON', 8, 4, 32, 31, FALSE, 12, 9, 'RAIN', 50),
        (46, 8, '2025-10-25', '15:30', 'AFTERNOON', 2, 3, 42, 25, FALSE, 22, 18, 'CLEAR', 79),
        (47, 8, '2025-10-25', '15:30', 'AFTERNOON', 12, 10, 20, 17, FALSE, NULL, NULL, 'CLEAR', 62),
        (48, 8, '2025-10-25', '12:00', 'NOON', 7, 6, 24, 34, FALSE, 3, NULL, 'RAIN', 56),
        (49, 9, '2025-11-01', '15:30', 'AFTERNOON', 1, 5, 16, 9, FALSE, 14, NULL, 'CLEAR', 70),
        (50, 9, '2025-11-01', '19:30', 'NIGHT', 9, 8, 27, 14, FALSE, 5, 12, 'COLD', 42),
        (51, 9, '2025-11-01', '12:00', 'NOON', 2, 11, 29, 39, FALSE, 22, NULL, 'CLEAR', 69),
        (52, 9, '2025-11-01', '19:30', 'NIGHT', 4, 12, 38, 15, FALSE, 9, NULL, 'CLEAR', 81),
        (53, 9, '2025-11-01', '15:30', 'AFTERNOON', 7, 3, 29, 33, FALSE, 3, 18, 'CLEAR', 81),
        (54, 9, '2025-11-01', '12:00', 'NOON', 6, 10, 22, 21, FALSE, NULL, NULL, 'CLEAR', 69),
        (55, 10, '2025-11-08', '19:30', 'NIGHT', 8, 1, 27, 23, FALSE, 12, 14, 'COLD', 38),
        (56, 10, '2025-11-08', '15:30', 'AFTERNOON', 2, 5, 10, 3, FALSE, 22, NULL, 'CLEAR', 61),
        (57, 10, '2025-11-08', '15:30', 'AFTERNOON', 9, 12, 34, 21, FALSE, 5, NULL, 'RAIN', 50),
        (58, 10, '2025-11-08', '12:00', 'NOON', 7, 11, 36, 31, FALSE, 3, NULL, 'CLEAR', 74),
        (59, 10, '2025-11-08', '19:30', 'NIGHT', 6, 4, 26, 16, FALSE, NULL, 9, 'COLD', 45),
        (60, 10, '2025-11-08', '15:30', 'AFTERNOON', 3, 10, 41, 17, FALSE, 18, NULL, 'RAIN', 49),
        (61, 11, '2025-11-15', '19:30', 'NIGHT', 1, 2, 31, 28, TRUE, 14, 22, 'CLEAR', 66),
        (62, 11, '2025-11-15', '15:30', 'AFTERNOON', 12, 8, 29, 23, FALSE, NULL, 12, 'CLEAR', 80),
        (63, 11, '2025-11-15', '15:30', 'AFTERNOON', 5, 7, 12, 32, FALSE, NULL, 3, 'CLEAR', 63),
        (64, 11, '2025-11-15', '19:30', 'NIGHT', 6, 9, 32, 33, FALSE, NULL, 5, 'CLEAR', 79),
        (65, 11, '2025-11-15', '19:30', 'NIGHT', 10, 11, 15, 32, FALSE, NULL, NULL, 'COLD', 45),
        (66, 11, '2025-11-15', '15:30', 'AFTERNOON', 4, 3, 29, 33, FALSE, 9, 18, 'CLEAR', 65);

  CREATE OR REPLACE TABLE SATURDAY_OPS.RAW.TEAM_GAME_STATS (
    GAME_ID INT, TEAM_ID INT, IS_HOME BOOLEAN, PLAYS INT, PASS_ATT INT, RUSH_ATT INT,
    PASS_YDS INT, RUSH_YDS INT, EXPLOSIVE_PLAYS INT, FOURTH_DOWN_ATT INT, FOURTH_DOWN_CONV INT,
    TURNOVERS INT, TIME_OF_POSSESSION_SEC INT, RED_ZONE_TRIPS INT, RED_ZONE_TDS INT,
    PENALTIES INT, SECOND_HALF_PASS_ATT INT, SECOND_HALF_RUSH_ATT INT);
  INSERT INTO SATURDAY_OPS.RAW.TEAM_GAME_STATS SELECT * FROM VALUES
    (1, 1, TRUE, 64, 34, 30, 195, 132, 4, 1, 1, 3, 1827, 3, 3, 6, 20, 12),
        (1, 12, FALSE, 61, 25, 36, 72, 134, 3, 2, 0, 3, 1818, 3, 2, 5, 13, 18),
        (2, 7, TRUE, 66, 32, 34, 158, 148, 4, 3, 1, 0, 1884, 4, 4, 7, 16, 17),
        (2, 2, FALSE, 83, 41, 42, 163, 165, 8, 0, 0, 2, 1712, 4, 4, 5, 21, 21),
        (3, 8, TRUE, 60, 20, 40, 80, 204, 6, 2, 1, 1, 1926, 5, 3, 4, 10, 20),
        (3, 6, FALSE, 61, 21, 40, 105, 212, 3, 1, 1, 2, 2105, 5, 4, 2, 11, 20),
        (4, 5, TRUE, 61, 27, 34, 85, 77, 2, 1, 1, 0, 2003, 4, 4, 5, 14, 17),
        (4, 10, FALSE, 68, 28, 40, 123, 233, 3, 2, 0, 1, 1873, 2, 2, 3, 14, 20),
        (5, 3, TRUE, 81, 56, 25, 236, 102, 7, 1, 1, 1, 1562, 8, 6, 4, 28, 13),
        (5, 9, FALSE, 77, 51, 26, 242, 110, 10, 0, 0, 1, 1614, 4, 3, 10, 26, 13),
        (6, 4, TRUE, 82, 46, 36, 178, 141, 6, 1, 0, 0, 1677, 4, 3, 8, 23, 18),
        (6, 11, FALSE, 73, 45, 28, 200, 96, 7, 0, 0, 3, 1665, 3, 3, 5, 23, 14),
        (7, 1, TRUE, 69, 36, 33, 176, 138, 7, 1, 0, 2, 1809, 4, 3, 5, 21, 14),
        (7, 7, FALSE, 66, 34, 32, 157, 146, 4, 0, 0, 0, 1760, 5, 3, 8, 17, 16),
        (8, 6, TRUE, 65, 21, 44, 83, 228, 5, 1, 1, 1, 1988, 1, 1, 2, 11, 22),
        (8, 12, FALSE, 55, 22, 33, 69, 95, 3, 2, 1, 2, 1800, 3, 2, 7, 11, 17),
        (9, 10, TRUE, 71, 26, 45, 91, 221, 3, 1, 1, 2, 1884, 3, 3, 5, 13, 23),
        (9, 2, FALSE, 91, 48, 43, 219, 211, 9, 1, 0, 1, 1729, 5, 4, 6, 24, 22),
        (10, 8, TRUE, 62, 20, 42, 78, 206, 2, 5, 4, 0, 1903, 2, 2, 7, 10, 21),
        (10, 3, FALSE, 75, 53, 22, 266, 98, 7, 1, 1, 2, 1659, 5, 4, 9, 27, 11),
        (11, 4, TRUE, 81, 42, 39, 186, 159, 8, 1, 1, 2, 1726, 2, 2, 5, 21, 20),
        (11, 5, FALSE, 56, 22, 34, 80, 168, 2, 0, 0, 2, 1777, 2, 2, 7, 11, 17),
        (12, 9, TRUE, 74, 50, 24, 223, 83, 10, 1, 0, 2, 1735, 6, 5, 11, 25, 12),
        (12, 11, FALSE, 77, 48, 29, 239, 83, 9, 3, 2, 2, 1814, 5, 5, 9, 24, 15),
        (13, 6, TRUE, 65, 24, 41, 81, 191, 4, 1, 0, 1, 1867, 3, 2, 8, 12, 21),
        (13, 1, FALSE, 66, 36, 30, 168, 147, 6, 2, 1, 1, 1879, 2, 2, 9, 21, 12),
        (14, 10, TRUE, 64, 24, 40, 98, 188, 3, 1, 1, 1, 2013, 3, 3, 5, 12, 20),
        (14, 7, FALSE, 63, 33, 30, 154, 112, 5, 2, 2, 1, 1917, 4, 3, 3, 17, 15),
        (15, 3, TRUE, 71, 49, 22, 199, 76, 6, 2, 0, 0, 1833, 3, 3, 5, 25, 11),
        (15, 12, FALSE, 63, 25, 38, 102, 149, 0, 1, 1, 1, 1736, 4, 2, 10, 13, 19),
        (16, 2, TRUE, 81, 41, 40, 188, 198, 8, 3, 2, 2, 1780, 4, 4, 6, 21, 20),
        (16, 4, FALSE, 91, 48, 43, 210, 167, 7, 2, 1, 1, 1767, 4, 3, 5, 24, 22),
        (17, 11, TRUE, 77, 51, 26, 259, 94, 7, 1, 1, 0, 1697, 3, 3, 5, 26, 13),
        (17, 8, FALSE, 66, 20, 46, 69, 264, 5, 3, 3, 2, 2193, 3, 3, 4, 10, 23),
        (18, 5, TRUE, 63, 26, 37, 86, 116, 2, 1, 0, 1, 1875, 2, 1, 6, 13, 19),
        (18, 9, FALSE, 80, 52, 28, 223, 101, 8, 2, 2, 0, 1635, 5, 4, 7, 26, 14),
        (19, 1, TRUE, 69, 35, 34, 161, 133, 6, 1, 1, 0, 1712, 2, 1, 6, 20, 15),
        (19, 10, FALSE, 64, 18, 46, 83, 188, 2, 2, 2, 0, 2056, 1, 1, 5, 9, 23),
        (20, 3, TRUE, 76, 48, 28, 220, 123, 8, 0, 0, 1, 1599, 3, 3, 10, 24, 14),
        (20, 6, FALSE, 69, 25, 44, 109, 247, 3, 0, 0, 2, 1858, 2, 2, 3, 13, 22),
        (21, 7, TRUE, 79, 41, 38, 178, 148, 4, 0, 0, 3, 1826, 3, 3, 6, 21, 19),
        (21, 4, FALSE, 77, 42, 35, 184, 163, 8, 0, 0, 0, 1772, 3, 3, 6, 21, 18),
        (22, 11, TRUE, 83, 55, 28, 242, 109, 9, 0, 0, 2, 1587, 3, 3, 5, 28, 14),
        (22, 12, FALSE, 62, 28, 34, 108, 156, 3, 2, 0, 1, 1786, 2, 2, 3, 14, 17),
        (23, 2, TRUE, 77, 41, 36, 175, 176, 7, 2, 1, 1, 1821, 4, 3, 9, 21, 18),
        (23, 9, FALSE, 70, 44, 26, 220, 104, 8, 0, 0, 0, 1769, 5, 4, 8, 22, 13),
        (24, 5, TRUE, 54, 20, 34, 70, 137, 3, 1, 0, 1, 1785, 3, 3, 8, 10, 17),
        (24, 8, FALSE, 66, 22, 44, 82, 214, 0, 4, 3, 1, 1960, 3, 2, 2, 11, 22),
        (25, 3, TRUE, 81, 51, 30, 226, 92, 7, 3, 1, 2, 1649, 6, 5, 4, 26, 15),
        (25, 1, FALSE, 69, 36, 33, 142, 184, 4, 1, 1, 2, 1858, 4, 3, 5, 21, 14),
        (26, 10, TRUE, 64, 20, 44, 68, 196, 4, 1, 0, 2, 2017, 3, 3, 9, 10, 22),
        (26, 4, FALSE, 87, 43, 44, 214, 220, 7, 0, 0, 2, 1898, 2, 2, 9, 22, 22),
        (27, 6, TRUE, 66, 20, 46, 101, 247, 3, 1, 1, 0, 2091, 4, 4, 6, 10, 23),
        (27, 11, FALSE, 72, 46, 26, 240, 97, 9, 1, 0, 2, 1707, 4, 2, 7, 23, 13),
        (28, 9, TRUE, 78, 48, 30, 177, 135, 9, 1, 0, 3, 1845, 5, 4, 7, 24, 15),
        (28, 7, FALSE, 67, 33, 34, 122, 145, 5, 1, 0, 1, 1776, 5, 4, 3, 17, 17),
        (29, 5, TRUE, 66, 24, 42, 91, 148, 3, 1, 0, 1, 1763, 1, 1, 5, 12, 21),
        (29, 12, FALSE, 60, 26, 34, 102, 112, 1, 1, 1, 1, 1814, 1, 1, 2, 13, 17),
        (30, 8, TRUE, 64, 22, 42, 95, 180, 4, 4, 1, 0, 2062, 4, 4, 6, 11, 21),
        (30, 2, FALSE, 85, 45, 40, 171, 153, 7, 0, 0, 0, 1793, 3, 3, 6, 23, 20),
        (31, 4, TRUE, 77, 40, 37, 183, 166, 6, 2, 1, 1, 1779, 4, 4, 5, 20, 19),
        (31, 1, FALSE, 67, 37, 30, 168, 129, 4, 1, 0, 1, 1658, 5, 4, 6, 22, 12),
        (32, 11, TRUE, 77, 50, 27, 249, 85, 6, 0, 0, 0, 1653, 3, 3, 7, 25, 14),
        (32, 3, FALSE, 72, 48, 24, 210, 73, 9, 1, 1, 2, 1670, 5, 4, 7, 24, 12),
        (33, 10, TRUE, 62, 20, 42, 73, 295, 3, 2, 2, 1, 2135, 4, 3, 4, 10, 21),
        (33, 9, FALSE, 79, 53, 26, 259, 97, 9, 0, 0, 1, 1747, 5, 4, 9, 27, 13),
        (34, 5, TRUE, 58, 22, 36, 85, 129, 2, 1, 1, 2, 1976, 3, 2, 4, 11, 18),
        (34, 6, FALSE, 63, 23, 40, 104, 206, 2, 0, 0, 0, 1831, 4, 3, 6, 12, 20),
        (35, 8, TRUE, 64, 20, 44, 61, 256, 4, 4, 1, 2, 2125, 4, 3, 7, 10, 22),
        (35, 7, FALSE, 67, 33, 34, 146, 147, 3, 2, 0, 3, 1806, 4, 3, 4, 17, 17),
        (36, 12, TRUE, 55, 21, 34, 92, 146, 4, 2, 2, 2, 1814, 4, 4, 7, 11, 17),
        (36, 2, FALSE, 77, 40, 37, 166, 193, 6, 4, 3, 1, 1760, 1, 1, 4, 20, 19),
        (37, 1, TRUE, 68, 36, 32, 142, 149, 7, 1, 1, 4, 1760, 3, 2, 6, 20, 14),
        (37, 11, FALSE, 78, 46, 32, 238, 121, 7, 1, 1, 2, 1614, 4, 4, 8, 23, 16),
        (38, 4, TRUE, 80, 42, 38, 164, 240, 7, 0, 0, 0, 1754, 1, 1, 8, 21, 19),
        (38, 9, FALSE, 75, 51, 24, 187, 111, 11, 2, 1, 2, 1545, 3, 3, 6, 26, 12),
        (39, 5, TRUE, 65, 26, 39, 106, 157, 0, 2, 1, 0, 1815, 4, 3, 8, 13, 20),
        (39, 3, FALSE, 82, 54, 28, 270, 101, 7, 1, 0, 1, 1611, 5, 4, 6, 27, 14),
        (40, 10, TRUE, 61, 21, 40, 98, 195, 3, 1, 0, 1, 1664, 2, 2, 9, 11, 20),
        (40, 8, FALSE, 65, 27, 38, 128, 193, 6, 4, 4, 0, 1813, 3, 3, 7, 14, 19),
        (41, 2, TRUE, 85, 43, 42, 190, 222, 9, 3, 1, 2, 1697, 4, 3, 8, 22, 21),
        (41, 6, FALSE, 74, 26, 48, 94, 220, 2, 2, 1, 2, 1853, 3, 3, 6, 13, 24),
        (42, 12, TRUE, 61, 27, 34, 91, 155, 3, 1, 1, 0, 1997, 4, 3, 9, 14, 17),
        (42, 7, FALSE, 67, 34, 33, 144, 188, 5, 2, 2, 1, 1864, 3, 2, 8, 17, 17),
        (43, 9, TRUE, 80, 52, 28, 286, 135, 11, 1, 0, 0, 1588, 6, 4, 5, 26, 14),
        (43, 1, FALSE, 71, 38, 33, 167, 157, 6, 1, 1, 1, 1864, 3, 2, 5, 23, 13),
        (44, 5, TRUE, 57, 23, 34, 99, 135, 1, 1, 1, 2, 2038, 2, 2, 6, 12, 17),
        (44, 11, FALSE, 68, 44, 24, 182, 80, 7, 2, 0, 1, 1763, 4, 3, 4, 22, 12),
        (45, 8, TRUE, 60, 14, 46, 47, 214, 2, 3, 3, 1, 2074, 4, 4, 3, 7, 23),
        (45, 4, FALSE, 79, 37, 42, 154, 197, 6, 1, 1, 0, 1745, 5, 3, 9, 19, 21),
        (46, 2, TRUE, 76, 44, 32, 211, 142, 7, 2, 1, 2, 1785, 6, 5, 9, 22, 16),
        (46, 3, FALSE, 81, 55, 26, 268, 101, 9, 2, 1, 1, 1831, 4, 3, 9, 28, 13),
        (47, 12, TRUE, 58, 22, 36, 65, 158, 3, 0, 0, 1, 2041, 3, 2, 5, 11, 18),
        (47, 10, FALSE, 64, 22, 42, 92, 240, 6, 2, 2, 1, 1834, 3, 2, 6, 11, 21),
        (48, 7, TRUE, 71, 32, 39, 125, 185, 5, 1, 1, 0, 1925, 4, 3, 4, 16, 20),
        (48, 6, FALSE, 61, 17, 44, 72, 264, 2, 0, 0, 2, 2063, 5, 4, 6, 9, 22),
        (49, 1, TRUE, 74, 39, 35, 158, 163, 4, 2, 2, 1, 1833, 2, 2, 7, 23, 14),
        (49, 5, FALSE, 55, 23, 32, 90, 147, 4, 2, 1, 1, 1966, 1, 1, 5, 12, 16),
        (50, 9, TRUE, 78, 52, 26, 230, 108, 11, 2, 2, 2, 1748, 4, 3, 7, 26, 13),
        (50, 8, FALSE, 64, 20, 44, 99, 227, 4, 5, 2, 0, 1969, 2, 2, 4, 10, 22),
        (51, 2, TRUE, 76, 40, 36, 158, 165, 6, 2, 2, 2, 1626, 5, 3, 4, 20, 18),
        (51, 11, FALSE, 77, 49, 28, 212, 93, 8, 2, 1, 2, 1510, 4, 4, 10, 25, 14),
        (52, 4, TRUE, 87, 48, 39, 229, 180, 5, 0, 0, 2, 1756, 6, 4, 9, 24, 20),
        (52, 12, FALSE, 56, 24, 32, 83, 122, 2, 2, 0, 1, 2197, 3, 2, 10, 12, 16),
        (53, 7, TRUE, 72, 34, 38, 187, 167, 4, 1, 0, 2, 1815, 4, 3, 9, 17, 19),
        (53, 3, FALSE, 75, 42, 33, 240, 176, 7, 1, 0, 0, 1675, 5, 4, 7, 21, 17),
        (54, 6, TRUE, 59, 20, 39, 82, 159, 3, 1, 0, 2, 2051, 3, 2, 6, 10, 20),
        (54, 10, FALSE, 63, 21, 42, 90, 198, 3, 0, 0, 1, 1898, 2, 2, 5, 11, 21),
        (55, 8, TRUE, 69, 23, 46, 71, 253, 4, 5, 2, 1, 1783, 3, 3, 6, 12, 23),
        (55, 1, FALSE, 66, 35, 31, 162, 122, 3, 1, 1, 1, 1783, 3, 3, 7, 19, 14),
        (56, 2, TRUE, 86, 48, 38, 152, 169, 6, 2, 2, 4, 1781, 1, 1, 5, 24, 19),
        (56, 5, FALSE, 64, 24, 40, 99, 194, 1, 1, 1, 2, 1938, 1, 0, 6, 12, 20),
        (57, 9, TRUE, 75, 47, 28, 233, 95, 11, 1, 1, 3, 1759, 3, 3, 4, 24, 14),
        (57, 12, FALSE, 64, 24, 40, 96, 140, 1, 3, 2, 0, 1906, 2, 2, 3, 12, 20),
        (58, 7, TRUE, 68, 36, 32, 163, 149, 6, 1, 0, 4, 1508, 5, 4, 5, 18, 16),
        (58, 11, FALSE, 75, 46, 29, 216, 126, 10, 1, 0, 2, 1598, 3, 3, 3, 23, 15),
        (59, 6, TRUE, 62, 20, 42, 68, 227, 1, 1, 0, 0, 2163, 4, 3, 6, 10, 21),
        (59, 4, FALSE, 83, 43, 40, 167, 146, 6, 1, 1, 1, 1663, 2, 2, 3, 22, 20),
        (60, 3, TRUE, 80, 48, 32, 252, 128, 8, 1, 1, 4, 1771, 5, 5, 4, 24, 16),
        (60, 10, FALSE, 65, 18, 47, 82, 270, 5, 2, 1, 3, 2122, 3, 2, 7, 9, 24),
        (61, 1, TRUE, 72, 40, 32, 179, 106, 6, 1, 0, 1, 2036, 4, 3, 8, 23, 13),
        (61, 2, FALSE, 85, 45, 40, 179, 189, 8, 2, 1, 2, 2098, 4, 3, 8, 23, 20),
        (62, 12, TRUE, 58, 26, 32, 101, 127, 2, 0, 0, 3, 1773, 4, 3, 8, 13, 16),
        (62, 8, FALSE, 70, 24, 46, 120, 276, 4, 3, 2, 0, 2000, 2, 2, 7, 12, 23),
        (63, 5, TRUE, 63, 28, 35, 116, 159, 3, 1, 0, 1, 1922, 1, 1, 5, 14, 18),
        (63, 7, FALSE, 69, 37, 32, 167, 147, 6, 1, 1, 0, 2000, 5, 4, 6, 19, 16),
        (64, 6, TRUE, 65, 25, 40, 99, 191, 6, 1, 0, 2, 1741, 4, 4, 8, 13, 20),
        (64, 9, FALSE, 78, 48, 30, 219, 124, 12, 2, 1, 1, 1754, 5, 4, 8, 24, 15),
        (65, 10, TRUE, 66, 26, 40, 102, 228, 5, 1, 1, 2, 1722, 2, 2, 5, 13, 20),
        (65, 11, FALSE, 86, 58, 28, 313, 138, 9, 1, 0, 2, 1716, 6, 4, 3, 29, 14),
        (66, 4, TRUE, 82, 44, 38, 217, 210, 8, 1, 0, 0, 1914, 5, 3, 6, 22, 19),
        (66, 3, FALSE, 74, 52, 22, 223, 73, 9, 1, 0, 1, 1481, 4, 4, 4, 26, 11);

  ---------------------------------------------------------------------------
  -- Game-day business drivers for UCLA home games (temporary, not exposed)
  ---------------------------------------------------------------------------
  CREATE OR REPLACE TEMPORARY TABLE SATURDAY_OPS.RAW.GF AS
  WITH f AS (
    SELECT * FROM VALUES
      -- week, fill, scan, q4 mult, per-cap mult, price mult, merch w, donate w, survey w, ticket w,
      -- neg food, neg parking, neg seats, neg team, neg price
      (1,  0.80, 0.86, 0.35, 0.90, 1.00, 1.0, 1.0, 1.0, 1.2, 0.62, 0.30, 0.25, 0.20, 0.35),
      (2,  0.97, 0.95, 1.10, 1.15, 1.40, 1.2, 0.9, 1.0, 1.0, 0.25, 0.40, 0.20, 0.35, 0.40),
      (4,  0.90, 0.76, 0.70, 0.80, 1.10, 0.8, 1.1, 1.0, 1.3, 0.30, 0.50, 0.68, 0.20, 0.35),
      (7,  0.94, 0.93, 0.60, 0.95, 1.20, 0.6, 0.35, 1.4, 2.6, 0.30, 0.35, 0.20, 0.85, 0.45),
      (9,  0.79, 0.91, 0.90, 1.00, 1.00, 0.9, 1.0, 0.9, 0.8, 0.25, 0.30, 0.20, 0.25, 0.30),
      (11, 0.99, 0.97, 1.40, 1.35, 1.60, 2.0, 2.6, 1.1, 1.1, 0.25, 0.55, 0.20, 0.05, 0.50)
      AS v(WK, FILL, SCAN, Q4M, PCM, PRICEM, MW, DW, SW, TW, NF, NP, NS, NT, NPR)
  ), g AS (
    SELECT f.*, gm.GAME_ID, gm.GAME_DATE, gm.KICKOFF_TIME, gm.WEATHER,
           ROW_NUMBER() OVER (ORDER BY f.WK) - 1 AS IDX,
           f.FILL * f.SCAN * f.PCM AS CW
    FROM f JOIN SATURDAY_OPS.RAW.GAMES gm ON gm.SEASON_WEEK = f.WK AND gm.HOME_TEAM_ID = 1
  )
  SELECT g.*,
    SUM(CW) OVER (ORDER BY IDX) / SUM(CW) OVER () AS C_HI, (SUM(CW) OVER (ORDER BY IDX) - CW) / SUM(CW) OVER () AS C_LO,
    SUM(MW) OVER (ORDER BY IDX) / SUM(MW) OVER () AS M_HI, (SUM(MW) OVER (ORDER BY IDX) - MW) / SUM(MW) OVER () AS M_LO,
    SUM(DW) OVER (ORDER BY IDX) / SUM(DW) OVER () AS D_HI, (SUM(DW) OVER (ORDER BY IDX) - DW) / SUM(DW) OVER () AS D_LO,
    SUM(SW) OVER (ORDER BY IDX) / SUM(SW) OVER () AS S_HI, (SUM(SW) OVER (ORDER BY IDX) - SW) / SUM(SW) OVER () AS S_LO,
    SUM(TW) OVER (ORDER BY IDX) / SUM(TW) OVER () AS T_HI, (SUM(TW) OVER (ORDER BY IDX) - TW) / SUM(TW) OVER () AS T_LO
  FROM g;

  ---------------------------------------------------------------------------
  -- Fans
  ---------------------------------------------------------------------------
  CREATE OR REPLACE TABLE SATURDAY_OPS.RAW.FANS AS
  SELECT n AS FAN_ID,
    CASE WHEN n <= 15000 THEN 'SEASON_TICKET' WHEN n <= 30000 THEN 'SINGLE_GAME'
         WHEN n <= 40000 THEN 'ALUMNI' ELSE 'STUDENT' END AS FAN_SEGMENT,
    CASE WHEN n > 40000 THEN 'NONE'
         WHEN SATURDAY_OPS.RAW.RND(n,'tier') < CASE WHEN n > 30000 THEN 0.45 WHEN n <= 15000 THEN 0.40 ELSE 0.10 END
           THEN CASE WHEN SATURDAY_OPS.RAW.RND(n,'t2') < 0.15 THEN 'GOLD' WHEN SATURDAY_OPS.RAW.RND(n,'t2') < 0.45 THEN 'SILVER' ELSE 'BRONZE' END
         ELSE 'NONE' END AS DONOR_TIER,
    LPAD((90001 + FLOOR(SATURDAY_OPS.RAW.RND(n,'zip') * 1700)::INT)::STRING, 5, '0') AS HOME_ZIP,
    CASE WHEN n > 40000 THEN 1 + FLOOR(SATURDAY_OPS.RAW.RND(n,'yrs') * 4)::INT ELSE 1 + FLOOR(SATURDAY_OPS.RAW.RND(n,'yrs') * 40)::INT END AS YEARS_AS_FAN,
    CASE WHEN n > 40000 THEN '18-24'
         ELSE GET(ARRAY_CONSTRUCT('25-34','35-44','45-54','55-64','65+'), FLOOR(SATURDAY_OPS.RAW.RND(n,'age') * 5)::INT)::STRING END AS AGE_BAND
  FROM (SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) AS n FROM TABLE(GENERATOR(ROWCOUNT => 50000)));

  ---------------------------------------------------------------------------
  -- Ticket sales (one row per sold seat, Rose Bowl capacity 89,000)
  ---------------------------------------------------------------------------
  CREATE OR REPLACE TABLE SATURDAY_OPS.RAW.TICKET_SALES AS
  WITH seats AS (
    SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1 AS n FROM TABLE(GENERATOR(ROWCOUNT => 534000))
  ), s AS (
    SELECT n, MOD(n, 89000) AS seat, gf.*,
      CASE WHEN MOD(n,89000) < 30000 THEN 'LOWER' WHEN MOD(n,89000) < 80000 THEN 'UPPER'
           WHEN MOD(n,89000) < 84000 THEN 'PREMIUM' ELSE 'STUDENT' END AS tier
    FROM seats JOIN SATURDAY_OPS.RAW.GF gf ON gf.IDX = FLOOR(seats.n / 89000)
    WHERE SATURDAY_OPS.RAW.RND(n,'sell') < gf.FILL
  )
  SELECT n + 1 AS TICKET_ID, GAME_ID,
    CASE tier WHEN 'STUDENT' THEN 40001 + FLOOR(SATURDAY_OPS.RAW.RND(n,'fan') * 10000)::INT
              WHEN 'PREMIUM' THEN 30001 + FLOOR(SATURDAY_OPS.RAW.RND(n,'fan') * 10000)::INT
              ELSE 1 + FLOOR(SATURDAY_OPS.RAW.RND(n,'fan') * 40000)::INT END AS FAN_ID,
    CASE tier WHEN 'LOWER' THEN 'L' || LPAD((1 + FLOOR(seat / 1250))::STRING, 2, '0')
              WHEN 'UPPER' THEN 'U' || LPAD((1 + FLOOR((seat - 30000) / 2000))::STRING, 2, '0')
              WHEN 'PREMIUM' THEN 'CLUB' || (1 + FLOOR((seat - 80000) / 1000))::STRING
              ELSE 'STU' || (1 + FLOOR((seat - 84000) / 1250))::STRING END AS SECTION,
    tier AS SEAT_TIER,
    ROUND(PRICEM * CASE tier WHEN 'LOWER' THEN 85 + 55 * SATURDAY_OPS.RAW.RND(n,'p')
                             WHEN 'UPPER' THEN 35 + 35 * SATURDAY_OPS.RAW.RND(n,'p')
                             WHEN 'PREMIUM' THEN 250 + 150 * SATURDAY_OPS.RAW.RND(n,'p')
                             ELSE 15 END, 2) AS PRICE,
    CASE WHEN tier = 'STUDENT' THEN 'ONLINE'
         WHEN SATURDAY_OPS.RAW.RND(n,'ch') < 0.10 THEN 'BOX'
         WHEN SATURDAY_OPS.RAW.RND(n,'ch') < 0.10 + 0.12 * PRICEM THEN 'RESALE'
         ELSE 'ONLINE' END AS PURCHASE_CHANNEL,
    DATEADD(day, -1 * FLOOR(SATURDAY_OPS.RAW.RND(n,'pd') * 60)::INT, GAME_DATE) AS PURCHASE_DATE,
    SATURDAY_OPS.RAW.RND(n,'scan') < SCAN * CASE WHEN tier = 'STUDENT' THEN 0.92 ELSE 1 END AS SCANNED_FLAG,
    CASE WHEN SATURDAY_OPS.RAW.RND(n,'scan') < SCAN * CASE WHEN tier = 'STUDENT' THEN 0.92 ELSE 1 END
         THEN DATEADD(minute, -90 + FLOOR(SATURDAY_OPS.RAW.RND(n,'st') * 120)::INT,
                      TIMESTAMP_NTZ_FROM_PARTS(GAME_DATE, TO_TIME(KICKOFF_TIME))) END AS SCAN_TIME
  FROM s;

  ---------------------------------------------------------------------------
  -- Concession sales (~1M line items)
  ---------------------------------------------------------------------------
  CREATE OR REPLACE TABLE SATURDAY_OPS.RAW.CONCESSION_SALES AS
  WITH r AS (
    SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) AS n FROM TABLE(GENERATOR(ROWCOUNT => 1000000))
  ), x AS (
    SELECT r.n, gf.*,
      CASE WHEN SATURDAY_OPS.RAW.RND(r.n,'bev') < CASE WHEN gf.WEATHER = 'HEAT' THEN 0.68 ELSE 0.42 END
        THEN GET(ARRAY_CONSTRUCT(
          OBJECT_CONSTRUCT('i','Bottled Water','c','BEV','p',6.00),
          OBJECT_CONSTRUCT('i','Fountain Soda','c','BEV','p',7.50),
          OBJECT_CONSTRUCT('i','Lemonade','c','BEV','p',8.00),
          OBJECT_CONSTRUCT('i','Craft Beer','c','ALCOHOL','p',15.00),
          OBJECT_CONSTRUCT('i','Hard Seltzer','c','ALCOHOL','p',14.00),
          OBJECT_CONSTRUCT('i','Coffee','c','BEV','p',5.50)), FLOOR(SATURDAY_OPS.RAW.RND(r.n,'item') * 6)::INT)
        ELSE GET(ARRAY_CONSTRUCT(
          OBJECT_CONSTRUCT('i','Rose Bowl Dog','c','FOOD','p',9.50),
          OBJECT_CONSTRUCT('i','Loaded Nachos','c','FOOD','p',14.00),
          OBJECT_CONSTRUCT('i','Carne Asada Fries','c','FOOD','p',16.00),
          OBJECT_CONSTRUCT('i','Chicken Tenders','c','FOOD','p',13.50),
          OBJECT_CONSTRUCT('i','Soft Pretzel','c','SNACK','p',8.00),
          OBJECT_CONSTRUCT('i','Popcorn','c','SNACK','p',7.00),
          OBJECT_CONSTRUCT('i','Churro','c','SNACK','p',6.50)), FLOOR(SATURDAY_OPS.RAW.RND(r.n,'item') * 7)::INT)
      END AS item_obj
    FROM r JOIN SATURDAY_OPS.RAW.GF gf
      ON SATURDAY_OPS.RAW.RND(r.n,'game') >= gf.C_LO AND SATURDAY_OPS.RAW.RND(r.n,'game') < gf.C_HI
  )
  SELECT n AS TXN_ID, GAME_ID,
    CASE WHEN SATURDAY_OPS.RAW.RND(n,'card') < 0.5 THEN 1 + FLOOR(SATURDAY_OPS.RAW.RND(n,'cf') * 50000)::INT END AS FAN_ID,
    1 + FLOOR(SATURDAY_OPS.RAW.RND(n,'stand') * 40)::INT AS STAND_ID,
    item_obj:i::STRING AS ITEM, item_obj:c::STRING AS ITEM_CATEGORY,
    1 + FLOOR(SATURDAY_OPS.RAW.RND(n,'qty') * 2.4)::INT AS QTY,
    item_obj:p::NUMBER(6,2) AS UNIT_PRICE,
    CASE WHEN SATURDAY_OPS.RAW.RND(n,'q4') < 0.10 * Q4M THEN 'Q4'
         WHEN SATURDAY_OPS.RAW.RND(n,'q') < 0.24 THEN 'PRE'
         WHEN SATURDAY_OPS.RAW.RND(n,'q') < 0.48 THEN 'Q1'
         WHEN SATURDAY_OPS.RAW.RND(n,'q') < 0.82 THEN 'Q2'
         ELSE 'Q3' END AS GAME_QUARTER
  FROM x;

  ---------------------------------------------------------------------------
  -- Merch sales
  ---------------------------------------------------------------------------
  CREATE OR REPLACE TABLE SATURDAY_OPS.RAW.MERCH_SALES AS
  WITH r AS (
    SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) AS n FROM TABLE(GENERATOR(ROWCOUNT => 120000))
  ), x AS (
    SELECT r.n, gf.*,
      GET(ARRAY_CONSTRUCT(
        OBJECT_CONSTRUCT('i','UCLA QB1 Home Jersey','c','JERSEY','p',130),
        OBJECT_CONSTRUCT('i','UCLA RB1 Home Jersey','c','JERSEY','p',130),
        OBJECT_CONSTRUCT('i','UCLA WR1 Away Jersey','c','JERSEY','p',130),
        OBJECT_CONSTRUCT('i','UCLA LB1 Alternate Jersey','c','JERSEY','p',140),
        OBJECT_CONSTRUCT('i','Bruins Snapback Hat','c','HAT','p',35),
        OBJECT_CONSTRUCT('i','Bruins Bucket Hat','c','HAT','p',32),
        OBJECT_CONSTRUCT('i','Rose Bowl Hoodie','c','APPAREL','p',75),
        OBJECT_CONSTRUCT('i','Bruins Gameday Tee','c','APPAREL','p',30),
        OBJECT_CONSTRUCT('i','Bruins Rally Towel','c','COLLECTIBLE','p',12),
        OBJECT_CONSTRUCT('i','Mini Helmet','c','COLLECTIBLE','p',40)), FLOOR(SATURDAY_OPS.RAW.RND(r.n,'prod') * 10)::INT) AS p
    FROM r JOIN SATURDAY_OPS.RAW.GF gf
      ON SATURDAY_OPS.RAW.RND(r.n,'game') >= gf.M_LO AND SATURDAY_OPS.RAW.RND(r.n,'game') < gf.M_HI
  )
  SELECT n AS ORDER_ID, GAME_ID, 1 + FLOOR(SATURDAY_OPS.RAW.RND(n,'fan') * 50000)::INT AS FAN_ID,
    p:i::STRING AS PRODUCT, p:c::STRING AS PRODUCT_CATEGORY,
    IFF(SATURDAY_OPS.RAW.RND(n,'ch') < 0.55, 'STADIUM', 'ONLINE') AS CHANNEL,
    1 + FLOOR(SATURDAY_OPS.RAW.RND(n,'qty') * 1.6)::INT AS QTY,
    p:p::NUMBER(8,2) AS UNIT_PRICE,
    ROUND(p:p::NUMBER(8,2) * 0.42, 2) AS UNIT_COST,
    IFF(SATURDAY_OPS.RAW.RND(n,'ch') < 0.55, GAME_DATE, DATEADD(day, FLOOR(SATURDAY_OPS.RAW.RND(n,'od') * 7)::INT, GAME_DATE)) AS ORDER_DATE
  FROM x;

  ---------------------------------------------------------------------------
  -- Donations: baseline giving plus post-game spikes
  ---------------------------------------------------------------------------
  CREATE OR REPLACE TABLE SATURDAY_OPS.RAW.DONATIONS AS
  WITH r AS (
    SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) AS n FROM TABLE(GENERATOR(ROWCOUNT => 15000))
  ), base AS (
    SELECT n, DATEADD(day, FLOOR(SATURDAY_OPS.RAW.RND(n,'d') * 136)::INT, '2025-08-01'::DATE) AS GIFT_DATE, 1.0 AS MULT
    FROM r WHERE n <= 6000
    UNION ALL
    SELECT n, DATEADD(day, FLOOR(SATURDAY_OPS.RAW.RND(n,'d') * 7)::INT, gf.GAME_DATE), IFF(gf.WK = 11, 1.6, 1.0)
    FROM r JOIN SATURDAY_OPS.RAW.GF gf
      ON SATURDAY_OPS.RAW.RND(r.n,'game') >= gf.D_LO AND SATURDAY_OPS.RAW.RND(r.n,'game') < gf.D_HI
    WHERE n > 6000
  )
  SELECT n AS GIFT_ID,
    CASE WHEN SATURDAY_OPS.RAW.RND(n,'seg') < 0.6 THEN 30001 + FLOOR(SATURDAY_OPS.RAW.RND(n,'fan') * 10000)::INT
         ELSE 1 + FLOOR(SATURDAY_OPS.RAW.RND(n,'fan') * 15000)::INT END AS FAN_ID,
    GIFT_DATE,
    ROUND(MULT * 25 * POWER(10, 2.4 * SATURDAY_OPS.RAW.RND(n,'amt') * SATURDAY_OPS.RAW.RND(n,'amt2')), 2) AS AMOUNT,
    CASE WHEN SATURDAY_OPS.RAW.RND(n,'fund') < 0.40 THEN 'ATHLETICS'
         WHEN SATURDAY_OPS.RAW.RND(n,'fund') < 0.65 THEN 'NIL_COLLECTIVE'
         WHEN SATURDAY_OPS.RAW.RND(n,'fund') < 0.85 THEN 'SCHOLARSHIP'
         ELSE 'FACILITIES' END AS FUND
  FROM base;

  ---------------------------------------------------------------------------
  -- Unstructured text 1: game recaps (one per team per game)
  ---------------------------------------------------------------------------
  CREATE OR REPLACE TABLE SATURDAY_OPS.RAW.GAME_RECAPS AS
  WITH b AS (
    SELECT s.*, g.SEASON_WEEK, g.WEATHER, g.TEMP_F, g.IS_RIVALRY,
      t.SCHOOL_NAME AS TEAM, t.MASCOT, t.TRUE_STYLE AS STYLE, o.SCHOOL_NAME AS OPP,
      IFF(s.IS_HOME, g.HOME_SCORE, g.AWAY_SCORE) AS PTS, IFF(s.IS_HOME, g.AWAY_SCORE, g.HOME_SCORE) AS OPP_PTS,
      IFF(s.IS_HOME, t.STADIUM, o.STADIUM) AS VENUE,
      s.GAME_ID * 100 + s.TEAM_ID AS k
    FROM SATURDAY_OPS.RAW.TEAM_GAME_STATS s
    JOIN SATURDAY_OPS.RAW.GAMES g ON g.GAME_ID = s.GAME_ID
    JOIN SATURDAY_OPS.RAW.TEAMS t ON t.TEAM_ID = s.TEAM_ID
    JOIN SATURDAY_OPS.RAW.TEAMS o ON o.TEAM_ID = IFF(s.IS_HOME, g.AWAY_TEAM_ID, g.HOME_TEAM_ID)
  )
  SELECT ROW_NUMBER() OVER (ORDER BY GAME_ID, TEAM_ID) AS RECAP_ID, GAME_ID, TEAM_ID,
    IFF(PTS > OPP_PTS,
        TEAM || ' ' || GET(ARRAY_CONSTRUCT('tops','outlasts','handles','gets past'), FLOOR(SATURDAY_OPS.RAW.RND(k,'h') * 4)::INT)::STRING || ' ' || OPP || ' ' || PTS || '-' || OPP_PTS,
        OPP || ' ' || GET(ARRAY_CONSTRUCT('hands','deals','sends home'), FLOOR(SATURDAY_OPS.RAW.RND(k,'h') * 3)::INT)::STRING || ' ' || TEAM || ' a ' || OPP_PTS || '-' || PTS || ' loss') AS HEADLINE,
    'Staff Writer' AS AUTHOR,
    ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
      'In Week ' || SEASON_WEEK || ' at ' || VENUE || ', the ' || MASCOT || IFF(PTS > OPP_PTS, ' beat ', ' fell to ') || OPP || ' ' || PTS || '-' || OPP_PTS || '.',
      CASE WEATHER WHEN 'RAIN' THEN 'Steady rain soaked the field all afternoon and both offenses struggled to hold onto the ball.'
                   WHEN 'HEAT' THEN 'With temperatures near ' || TEMP_F || ' degrees at kickoff, the heat was a factor for both sidelines.'
                   WHEN 'COLD' THEN 'A cold ' || TEMP_F || '-degree night tested ball security on both sides.' END,
      IFF(IS_RIVALRY, 'Rivalry week delivered the kind of atmosphere this matchup is known for.', NULL),
      CASE STYLE
        WHEN 'AIR_RAID' THEN GET(ARRAY_CONSTRUCT(
          'The ' || MASCOT || ' spread the field with four and five receiver sets and put the ball in the air on nearly every down.',
          'It was an air raid clinic: empty backfields, quick throws and deep shots down the sideline.',
          'The ' || MASCOT || ' lived in the shotgun and attacked through the air from the opening snap.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'s') * 3)::INT)::STRING
        WHEN 'TEMPO_SPREAD' THEN GET(ARRAY_CONSTRUCT(
          'The ' || MASCOT || ' played at a breakneck pace, snapping the ball before the defense could substitute.',
          'Tempo was the story. The ' || MASCOT || ' went no-huddle and wore the defense down with a blur of spread looks.',
          'Running a hurry-up spread attack, the ' || MASCOT || ' piled up ' || PLAYS || ' snaps.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'s') * 3)::INT)::STRING
        WHEN 'GROUND_AND_POUND' THEN GET(ARRAY_CONSTRUCT(
          'The ' || MASCOT || ' leaned on a physical offensive line and pounded the ball between the tackles.',
          'It was ground and pound football: heavy formations, downhill runs and long, clock-eating drives.',
          'The ' || MASCOT || ' ran the ball ' || RUSH_ATT || ' times and dared the defense to stop it.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'s') * 3)::INT)::STRING
        WHEN 'DEFENSIVE_GRIND' THEN GET(ARRAY_CONSTRUCT(
          'This was a defensive grind. The ' || MASCOT || ' won with field position, punting and a defense that refused to break.',
          'Points were scarce as the ' || MASCOT || ' played conservative offense and leaned on their defense.',
          'The ' || MASCOT || ' kept it close to the vest, trusting the defense and the punt team to win the hidden yardage battle.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'s') * 3)::INT)::STRING
        ELSE GET(ARRAY_CONSTRUCT(
          'The ' || MASCOT || ' mixed it up, taking what the defense gave them on the ground and through the air.',
          'A balanced attack kept the defense guessing, with no clear tendency between run and pass.',
          'The ' || MASCOT || ' showed a balanced offense, splitting carries and targets evenly.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'s') * 3)::INT)::STRING
      END,
      TEAM || ' threw ' || PASS_ATT || ' passes for ' || PASS_YDS || ' yards and ran ' || RUSH_ATT || ' times for ' || RUSH_YDS || ' yards.',
      'After halftime the offense called ' || SECOND_HALF_PASS_ATT || ' passes and ' || SECOND_HALF_RUSH_ATT || ' runs.',
      CASE WHEN FOURTH_DOWN_ATT >= 3 THEN 'The ' || MASCOT || ' went for it on fourth down ' || FOURTH_DOWN_ATT || ' times, converting ' || FOURTH_DOWN_CONV || '.'
           WHEN EXPLOSIVE_PLAYS >= 9 THEN 'They ripped off ' || EXPLOSIVE_PLAYS || ' plays of 20 yards or more.'
           WHEN TURNOVERS >= 3 THEN 'Turnovers were costly, with ' || TURNOVERS || ' giveaways.'
           ELSE 'They held the ball for ' || ROUND(TIME_OF_POSSESSION_SEC / 60) || ' minutes.' END,
      'The turning point came when ' || GET(ARRAY_CONSTRUCT('QB1','RB1','WR1','WR2','TE1','LB1','CB1','S1','DE1'), FLOOR(SATURDAY_OPS.RAW.RND(k,'pl') * 9)::INT)::STRING || ' '
        || IFF(PTS > OPP_PTS,
             GET(ARRAY_CONSTRUCT('broke loose for a long touchdown in the fourth quarter.','made a game-sealing play late.','delivered on a critical third down to keep the final drive alive.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'tp') * 3)::INT)::STRING,
             GET(ARRAY_CONSTRUCT('was stopped short on a key fourth-quarter snap.','was flagged on a drive-killing penalty.','could not hold on to a potential go-ahead score.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'tp') * 3)::INT)::STRING)
    ), ' ') AS BODY
  FROM b;

  ---------------------------------------------------------------------------
  -- Unstructured text 2: coach press conference transcripts
  -- (planted mismatch: UCLA claims run-first while its second-half pass rate is ~62%)
  ---------------------------------------------------------------------------
  CREATE OR REPLACE TABLE SATURDAY_OPS.RAW.PRESSER_TRANSCRIPTS AS
  WITH b AS (
    SELECT s.*, g.SEASON_WEEK, g.WEATHER, t.SCHOOL_NAME AS TEAM, t.COACH_ALIAS, t.TRUE_STYLE AS STYLE,
      o.SCHOOL_NAME AS OPP,
      IFF(s.IS_HOME, g.HOME_SCORE, g.AWAY_SCORE) AS PTS, IFF(s.IS_HOME, g.AWAY_SCORE, g.HOME_SCORE) AS OPP_PTS,
      s.GAME_ID * 100 + s.TEAM_ID AS k
    FROM SATURDAY_OPS.RAW.TEAM_GAME_STATS s
    JOIN SATURDAY_OPS.RAW.GAMES g ON g.GAME_ID = s.GAME_ID
    JOIN SATURDAY_OPS.RAW.TEAMS t ON t.TEAM_ID = s.TEAM_ID
    JOIN SATURDAY_OPS.RAW.TEAMS o ON o.TEAM_ID = IFF(s.IS_HOME, g.AWAY_TEAM_ID, g.HOME_TEAM_ID)
  )
  SELECT ROW_NUMBER() OVER (ORDER BY GAME_ID, TEAM_ID) AS PRESSER_ID, GAME_ID, TEAM_ID, COACH_ALIAS,
    ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
      COACH_ALIAS || ', Week ' || SEASON_WEEK || ' vs ' || OPP || '.',
      'Opening statement: ' || IFF(PTS > OPP_PTS,
        GET(ARRAY_CONSTRUCT('Proud of our guys. That was a complete team win.','Great win. We found a way and I love the resilience in that locker room.','That is a big win for this program and I am happy for our players.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'o') * 3)::INT)::STRING,
        GET(ARRAY_CONSTRUCT('That one hurts. We did not execute and that starts with me.','Credit to them. We have to coach better and play better.','Not good enough. We made too many mistakes to beat a good football team.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'o') * 3)::INT)::STRING),
      'Q: How would you describe your game plan today? A: ' ||
      CASE WHEN TEAM = 'UCLA' THEN GET(ARRAY_CONSTRUCT(
          'We are a run-first football team. Establishing the run is our identity and that is what we tried to do.',
          'Our plan is always to run the football, control the line of scrimmage and play off of that.',
          'We want to be physical and run the ball. That is who we are and we are not going to change.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'p') * 3)::INT)::STRING
        WHEN STYLE = 'AIR_RAID' THEN GET(ARRAY_CONSTRUCT(
          'We want to throw it around and make you defend every blade of grass.',
          'We are going to spread you out and throw it. Our quarterback can make every throw.',
          'Five wide, get the ball out fast and take our shots. That is our offense.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'p') * 3)::INT)::STRING
        WHEN STYLE = 'TEMPO_SPREAD' THEN GET(ARRAY_CONSTRUCT(
          'We wanted to play fast, snap it quick and wear them out with tempo.',
          'Pace is our weapon. If they cannot substitute, we have them where we want them.',
          'We play fast. More snaps means more chances for our playmakers.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'p') * 3)::INT)::STRING
        WHEN STYLE = 'GROUND_AND_POUND' THEN GET(ARRAY_CONSTRUCT(
          'We are going to line up and run the football. We want to impose our will up front.',
          'Our offensive line is the heart of this team. We run it, we run it again, and we wear you down.',
          'Physical football. Win the line of scrimmage and the rest takes care of itself.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'p') * 3)::INT)::STRING
        WHEN STYLE = 'DEFENSIVE_GRIND' THEN GET(ARRAY_CONSTRUCT(
          'Our defense travels. We win field position, we win the turnover battle and we grind it out.',
          'We do not need to be flashy. Play great defense, punt well and do not turn it over.',
          'Complementary football. Our defense sets the tone and the offense protects the ball.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'p') * 3)::INT)::STRING
        ELSE GET(ARRAY_CONSTRUCT(
          'We want to be balanced and take what the defense gives us.',
          'We have answers in the run game and the pass game. We just have to execute.',
          'Our plan was to stay balanced and not become predictable.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'p') * 3)::INT)::STRING
      END,
      'Q: Talk about the decisions on fourth down. A: ' ||
      CASE WHEN FOURTH_DOWN_ATT >= 3 THEN 'We are going to go for it on fourth down. We went for it ' || FOURTH_DOWN_ATT || ' times today because I trust our guys. That is who we are.'
           WHEN FOURTH_DOWN_ATT >= 1 THEN 'We went for it when the analytics said to. You have to be aggressive in those spots.'
           ELSE 'We played the field position game and trusted our punt team.' END,
      CASE WHEN SATURDAY_OPS.RAW.RND(k,'inj') < 0.35 THEN
        'Q: Any injury updates? A: ' || GET(ARRAY_CONSTRUCT('RB1','WR2','LB1','CB1','OL1'), FLOOR(SATURDAY_OPS.RAW.RND(k,'ip') * 5)::INT)::STRING || ' '
          || GET(ARRAY_CONSTRUCT('tweaked an ankle and we will know more Monday.','is day to day.','is in concussion protocol.'), FLOOR(SATURDAY_OPS.RAW.RND(k,'it') * 3)::INT)::STRING END,
      CASE WHEN IS_HOME THEN 'Q: What did the home crowd mean today? A: ' ||
        IFF(PTS > OPP_PTS, 'The crowd was electric. Our fans made a real difference.', 'Our fans deserved better today. We owe them one.') END,
      CASE WHEN WEATHER = 'RAIN' THEN 'Q: How did the rain affect things? A: Ball security was a priority. It was a sloppy day for both teams.' END
    ), ' ') AS TRANSCRIPT
  FROM b;

  ---------------------------------------------------------------------------
  -- Unstructured text 3: fan surveys (UCLA home games)
  ---------------------------------------------------------------------------
  CREATE OR REPLACE TABLE SATURDAY_OPS.RAW.FAN_SURVEYS AS
  WITH r AS (
    SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) AS n FROM TABLE(GENERATOR(ROWCOUNT => 4000))
  ), x AS (
    SELECT r.n, gf.GAME_ID, gf.GAME_DATE,
      SATURDAY_OPS.RAW.RND(r.n,'nf') < gf.NF AS neg_food,
      SATURDAY_OPS.RAW.RND(r.n,'np') < gf.NP AS neg_park,
      SATURDAY_OPS.RAW.RND(r.n,'ns') < gf.NS AS neg_seat,
      SATURDAY_OPS.RAW.RND(r.n,'nt') < gf.NT AS neg_team,
      SATURDAY_OPS.RAW.RND(r.n,'npr') < gf.NPR AS neg_price,
      gf.WEATHER
    FROM r JOIN SATURDAY_OPS.RAW.GF gf
      ON SATURDAY_OPS.RAW.RND(r.n,'game') >= gf.S_LO AND SATURDAY_OPS.RAW.RND(r.n,'game') < gf.S_HI
  )
  SELECT n AS SURVEY_ID, GAME_ID, 1 + FLOOR(SATURDAY_OPS.RAW.RND(n,'fan') * 50000)::INT AS FAN_ID,
    GREATEST(1, LEAST(10, ROUND(9 - 1.7 * (neg_food::INT + neg_park::INT + neg_seat::INT + 1.5 * neg_team::INT + neg_price::INT)
      + 2 * SATURDAY_OPS.RAW.RND(n,'rt') - 1))) AS OVERALL_RATING,
    ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
      IFF(neg_team,
        GET(ARRAY_CONSTRUCT('Brutal game to watch. The offense looked lost.','We got embarrassed on our own field.','Left at the end of the third quarter. Hard to watch.','The team did not show up today.'), FLOOR(SATURDAY_OPS.RAW.RND(n,'t') * 4)::INT)::STRING,
        GET(ARRAY_CONSTRUCT('Great energy from the team today.','Loved the effort from the Bruins.','What a game. The atmosphere was incredible.','The team played hard and it was fun to watch.'), FLOOR(SATURDAY_OPS.RAW.RND(n,'t') * 4)::INT)::STRING),
      CASE WHEN SATURDAY_OPS.RAW.RND(n,'mf') < 0.55 THEN IFF(neg_food,
        IFF(WEATHER = 'HEAT', GET(ARRAY_CONSTRUCT('The water lines were 30 minutes long in 98 degree heat.','Concessions ran out of water by the second quarter.'), FLOOR(SATURDAY_OPS.RAW.RND(n,'f') * 2)::INT)::STRING,
            GET(ARRAY_CONSTRUCT('Fourteen dollar nachos and they were cold.','Food lines were way too long at halftime.','The hot dog was overpriced and underwhelming.'), FLOOR(SATURDAY_OPS.RAW.RND(n,'f') * 3)::INT)::STRING),
        GET(ARRAY_CONSTRUCT('The carne asada fries were excellent.','Food lines moved quickly.','Loved the new food options.'), FLOOR(SATURDAY_OPS.RAW.RND(n,'f') * 3)::INT)::STRING) END,
      CASE WHEN SATURDAY_OPS.RAW.RND(n,'mp') < 0.55 THEN IFF(neg_park,
        GET(ARRAY_CONSTRUCT('Parking took over an hour to get out of.','The golf course lots were a muddy mess.','Shuttle wait was ridiculous after the game.'), FLOOR(SATURDAY_OPS.RAW.RND(n,'p') * 3)::INT)::STRING,
        GET(ARRAY_CONSTRUCT('Parking was smooth this time.','The shuttle was quick and easy.','Getting in and out was painless.'), FLOOR(SATURDAY_OPS.RAW.RND(n,'p') * 3)::INT)::STRING) END,
      CASE WHEN SATURDAY_OPS.RAW.RND(n,'ms') < 0.50 THEN IFF(neg_seat,
        IFF(WEATHER = 'RAIN', 'Our seats were soaked and there was no cover anywhere.',
            GET(ARRAY_CONSTRUCT('The bench seats are cramped and uncomfortable.','Our view was blocked the whole game.'), FLOOR(SATURDAY_OPS.RAW.RND(n,'s') * 2)::INT)::STRING),
        GET(ARRAY_CONSTRUCT('Great view from our seats.','Seats were clean and comfortable.'), FLOOR(SATURDAY_OPS.RAW.RND(n,'s') * 2)::INT)::STRING) END,
      CASE WHEN SATURDAY_OPS.RAW.RND(n,'mpr') < 0.45 THEN IFF(neg_price,
        GET(ARRAY_CONSTRUCT('Ticket prices are getting out of hand.','Way too expensive for a family of four.','Resale prices were insane for this one.'), FLOOR(SATURDAY_OPS.RAW.RND(n,'pr') * 3)::INT)::STRING,
        GET(ARRAY_CONSTRUCT('Good value for the experience.','Worth every penny.'), FLOOR(SATURDAY_OPS.RAW.RND(n,'pr') * 2)::INT)::STRING) END
    ), ' ') AS COMMENT
  FROM x;

  ---------------------------------------------------------------------------
  -- Unstructured text 4: fan service tickets (UCLA home games)
  ---------------------------------------------------------------------------
  CREATE OR REPLACE TABLE SATURDAY_OPS.RAW.FAN_SERVICE_TICKETS AS
  WITH r AS (
    SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) AS n FROM TABLE(GENERATOR(ROWCOUNT => 2000))
  ), x AS (
    SELECT r.n, gf.GAME_ID, gf.GAME_DATE, gf.WK, gf.WEATHER,
      CASE
        WHEN gf.WK = 7 AND SATURDAY_OPS.RAW.RND(r.n,'ov') < 0.40 THEN 'APP'
        WHEN gf.WEATHER = 'HEAT' AND SATURDAY_OPS.RAW.RND(r.n,'ov') < 0.40 THEN 'CONC'
        WHEN gf.WEATHER = 'RAIN' AND SATURDAY_OPS.RAW.RND(r.n,'ov') < 0.40 THEN 'SEAT'
        WHEN SATURDAY_OPS.RAW.RND(r.n,'cat') < 0.20 THEN 'PARK'
        WHEN SATURDAY_OPS.RAW.RND(r.n,'cat') < 0.35 THEN 'SEAT'
        WHEN SATURDAY_OPS.RAW.RND(r.n,'cat') < 0.50 THEN 'REF'
        WHEN SATURDAY_OPS.RAW.RND(r.n,'cat') < 0.58 THEN 'ADA'
        WHEN SATURDAY_OPS.RAW.RND(r.n,'cat') < 0.70 THEN 'CONC'
        WHEN SATURDAY_OPS.RAW.RND(r.n,'cat') < 0.90 THEN 'APP'
        ELSE 'SEC' END AS c,
      'L' || LPAD((1 + FLOOR(SATURDAY_OPS.RAW.RND(r.n,'sec') * 24)::INT)::STRING, 2, '0') AS sec,
      (1 + FLOOR(SATURDAY_OPS.RAW.RND(r.n,'gate') * 12)::INT)::STRING AS gate,
      FLOOR(SATURDAY_OPS.RAW.RND(r.n,'v') * 2)::INT AS v
    FROM r JOIN SATURDAY_OPS.RAW.GF gf
      ON SATURDAY_OPS.RAW.RND(r.n,'game') >= gf.T_LO AND SATURDAY_OPS.RAW.RND(r.n,'game') < gf.T_HI
  )
  SELECT n AS CASE_ID, GAME_ID, 1 + FLOOR(SATURDAY_OPS.RAW.RND(n,'fan') * 50000)::INT AS FAN_ID,
    DATEADD(minute, FLOOR(SATURDAY_OPS.RAW.RND(n,'ct') * 4320)::INT, GAME_DATE::TIMESTAMP_NTZ) AS CREATED_AT,
    GET(ARRAY_CONSTRUCT('EMAIL','CHAT','PHONE'), FLOOR(SATURDAY_OPS.RAW.RND(n,'chn') * 3)::INT)::STRING AS CHANNEL,
    CASE c WHEN 'PARK' THEN IFF(v = 0, 'Stuck in the lot for 90 minutes', 'Parking pass not honored')
           WHEN 'SEAT' THEN IFF(v = 0, 'Someone was in our seats', 'Seats were wet and damaged')
           WHEN 'REF' THEN IFF(v = 0, 'Requesting a refund', 'Charged twice for tickets')
           WHEN 'ADA' THEN IFF(v = 0, 'Wheelchair seating issue', 'Elevator out of service')
           WHEN 'CONC' THEN IFF(v = 0, 'Ran out of water', 'Wrong charge at a food stand')
           WHEN 'APP' THEN IFF(v = 0, 'Mobile tickets would not load', 'App crashed at the gate')
           ELSE IFF(v = 0, 'Bag policy confusion', 'Unruly fans near our section') END AS SUBJECT,
    CASE c
      WHEN 'PARK' THEN IFF(v = 0, 'We parked in Lot H and it took 90 minutes to exit after the game. Nobody was directing traffic near Gate ' || gate || '. This needs to be fixed.',
                                  'I bought a prepaid parking pass but the attendant said it was invalid. I had to pay again in cash. Please refund the second charge.')
      WHEN 'SEAT' THEN IFF(v = 0, 'When we got to section ' || sec || ' there were other people in our seats and no usher could help us for most of the first quarter.',
                                  'Our seats in section ' || sec || ' were soaked and one seat back was broken. We stood for most of the game.')
      WHEN 'REF' THEN IFF(v = 0, 'I could not attend due to a family emergency. Is there any way to get a refund or credit for my tickets in section ' || sec || '?',
                                 'My card was charged twice for the same order. Please refund the duplicate charge.')
      WHEN 'ADA' THEN IFF(v = 0, 'The accessible seating platform in section ' || sec || ' was full of people without wheelchairs. My father had nowhere to sit.',
                                 'The elevator near Gate ' || gate || ' was out of service and there was no alternate route for my mother who uses a walker.')
      WHEN 'CONC' THEN IFF(v = 0, 'The stands near Gate ' || gate || ' ran out of water before halftime. It was close to 100 degrees and the food and drink lines were 40 minutes long.',
                                  'I was charged for three meals but only ordered one at the stand behind section ' || sec || '.')
      WHEN 'APP' THEN IFF(v = 0, 'My mobile tickets would not load at Gate ' || gate || '. The line behind me grew and staff said the app was down for everyone.',
                                 'The ticketing app crashed right as I reached the scanner at Gate ' || gate || '. We missed kickoff waiting for a paper reprint.')
      ELSE IFF(v = 0, 'Security at Gate ' || gate || ' rejected my clear bag even though it met the posted policy.',
                      'A group near section ' || sec || ' was swearing and throwing things all game. Security never came.')
    END AS BODY,
    IFF(SATURDAY_OPS.RAW.RND(n,'st') < 0.7, 'CLOSED', 'OPEN') AS STATUS
  FROM x;

  DROP TABLE IF EXISTS SATURDAY_OPS.RAW.GF;
  RETURN 'Saturday Ops data generated';
END;
$$

In [ ]:
%%sql -r row_counts
SELECT TABLE_NAME, ROW_COUNT
FROM SATURDAY_OPS.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'RAW'
ORDER BY TABLE_NAME

Here is what the unstructured text looks like: one coach press conference and two fan service tickets.

In [ ]:
%%sql -r sample_text
SELECT 'PRESS CONFERENCE' AS SOURCE, TRANSCRIPT AS TEXT
FROM SATURDAY_OPS.RAW.PRESSER_TRANSCRIPTS WHERE PRESSER_ID = 2
UNION ALL
SELECT 'FAN SERVICE TICKET', SUBJECT || ': ' || BODY
FROM (SELECT * FROM SATURDAY_OPS.RAW.FAN_SERVICE_TICKETS ORDER BY CASE_ID LIMIT 2)

## 2. Cortex AI Functions: turn text into data
Business units cannot put a press conference or a fan complaint in a dashboard. AI Functions run inside SQL and turn text into columns.

| Function | Used for |
|---|---|
| `AI_SENTIMENT` | Overall and per-aspect sentiment (food, parking, seats, team, price) on fan surveys |
| `AI_CLASSIFY` | Route fan service tickets into categories; label each game recap with a play style |
| `AI_EXTRACT` | Pull section, gate and issue out of tickets; pull the stated game plan out of press conferences |
| `AI_SUMMARIZE_AGG` | Summarize a whole season of recaps into one team identity |

### 2a. Fan Experience: sentiment by aspect

In [ ]:
%%sql -r fan_insights
-- AI_SENTIMENT scores overall sentiment plus each aspect we care about.
CREATE OR REPLACE TABLE SATURDAY_OPS.ANALYTICS.FAN_INSIGHTS AS
WITH s AS (
  SELECT SURVEY_ID, GAME_ID, FAN_ID, OVERALL_RATING, COMMENT,
         AI_SENTIMENT(COMMENT, ['food', 'parking', 'seats', 'team', 'price']) AS R
  FROM SATURDAY_OPS.RAW.FAN_SURVEYS
)
SELECT SURVEY_ID, GAME_ID, FAN_ID, OVERALL_RATING, COMMENT,
  GET(FILTER(R:categories, c -> c:name = 'overall'), 0):sentiment::STRING AS OVERALL_SENTIMENT,
  GET(FILTER(R:categories, c -> c:name = 'food'), 0):sentiment::STRING    AS FOOD_SENTIMENT,
  GET(FILTER(R:categories, c -> c:name = 'parking'), 0):sentiment::STRING AS PARKING_SENTIMENT,
  GET(FILTER(R:categories, c -> c:name = 'seats'), 0):sentiment::STRING   AS SEATS_SENTIMENT,
  GET(FILTER(R:categories, c -> c:name = 'team'), 0):sentiment::STRING    AS TEAM_SENTIMENT,
  GET(FILTER(R:categories, c -> c:name = 'price'), 0):sentiment::STRING   AS PRICE_SENTIMENT,
  CASE GET(FILTER(R:categories, c -> c:name = 'overall'), 0):sentiment::STRING
    WHEN 'positive' THEN 1 WHEN 'negative' THEN -1 ELSE 0 END AS SENTIMENT_SCORE
FROM s

### 2b. Fan Experience: classify and extract service tickets

In [ ]:
%%sql -r ticket_insights
-- AI_CLASSIFY routes each fan service ticket; AI_EXTRACT pulls structured fields out of free text.
CREATE OR REPLACE TABLE SATURDAY_OPS.ANALYTICS.TICKET_INSIGHTS AS
WITH t AS (
  SELECT CASE_ID, GAME_ID, FAN_ID, CREATED_AT, STATUS, SUBJECT, BODY,
    AI_CLASSIFY(SUBJECT || '. ' || BODY,
      [{'label': 'PARKING', 'description': 'parking lots, traffic, shuttles, parking passes'},
       {'label': 'SEATING', 'description': 'seat location, someone in my seat, wet or broken seats'},
       {'label': 'REFUND', 'description': 'refund requests, double charges, billing for tickets'},
       {'label': 'ACCESSIBILITY', 'description': 'wheelchair seating, elevators, mobility access'},
       {'label': 'CONCESSIONS', 'description': 'food and drink stands, water, lines, wrong food charges'},
       {'label': 'TICKETING_APP', 'description': 'mobile tickets, ticketing app, scanners at the gate'},
       {'label': 'SECURITY', 'description': 'bag policy, unruly fans, safety staff'}]) AS C,
    AI_EXTRACT(text => BODY, responseFormat => {
      'section': 'Stadium section code such as L12 or CLUB2. Answer NONE if no section is mentioned.',
      'gate': 'Gate number as digits only. Answer NONE if no gate is mentioned.',
      'issue_summary': 'In 3 to 6 words, what went wrong?'}) AS E
  FROM SATURDAY_OPS.RAW.FAN_SERVICE_TICKETS
)
SELECT CASE_ID, GAME_ID, FAN_ID, CREATED_AT, STATUS, SUBJECT, BODY,
  C:labels[0]::STRING AS CATEGORY,
  IFF(UPPER(E:response:section::STRING) IN ('NONE', 'NULL', ''), NULL, E:response:section::STRING) AS SECTION,
  IFF(UPPER(E:response:gate::STRING) IN ('NONE', 'NULL', ''), NULL, E:response:gate::STRING) AS GATE,
  E:response:issue_summary::STRING AS ISSUE_SUMMARY
FROM t

### 2c. Football Ops: what do the media and the coaches say?

In [ ]:
%%sql -r game_text_insights
-- Media recaps -> play style label. Coach pressers -> stated game plan, run-first claim, tone.
CREATE OR REPLACE TABLE SATURDAY_OPS.ANALYTICS.GAME_TEXT_INSIGHTS AS
WITH r AS (
  SELECT GAME_ID, TEAM_ID, BODY,
    AI_CLASSIFY(BODY, [
      {'label': 'AIR_RAID', 'description': 'pass-heavy spread offense, throws on most downs'},
      {'label': 'GROUND_AND_POUND', 'description': 'run-heavy, physical, heavy formations, clock control'},
      {'label': 'TEMPO_SPREAD', 'description': 'no-huddle, fast pace, many snaps, spread formations'},
      {'label': 'DEFENSIVE_GRIND', 'description': 'conservative offense, low scoring, relies on defense and field position'},
      {'label': 'BALANCED', 'description': 'even mix of run and pass, no strong tendency'}]) AS C,
    AI_EXTRACT(text => BODY, responseFormat => {
      'key_player': 'Which player alias (such as QB1 or RB1) made the turning point play?',
      'turning_point': 'In under 12 words, what was the turning point?'}) AS E
  FROM SATURDAY_OPS.RAW.GAME_RECAPS
), p AS (
  SELECT GAME_ID, TEAM_ID, TRANSCRIPT,
    AI_EXTRACT(text => TRANSCRIPT, responseFormat => {
      'stated_game_plan': 'In under 15 words, what game plan or identity does the coach describe?',
      'claims_run_first': 'Does the coach say the team is run-first or wants to run the football? Answer YES or NO.'}) AS E,
    AI_SENTIMENT(TRANSCRIPT) AS S
  FROM SATURDAY_OPS.RAW.PRESSER_TRANSCRIPTS
)
SELECT r.GAME_ID, r.TEAM_ID,
  r.C:labels[0]::STRING AS MEDIA_STYLE_LABEL,
  r.E:response:key_player::STRING AS KEY_PLAYER,
  r.E:response:turning_point::STRING AS TURNING_POINT,
  p.E:response:stated_game_plan::STRING AS COACH_STATED_PLAN,
  UPPER(p.E:response:claims_run_first::STRING) LIKE 'YES%' AS COACH_CLAIMS_RUN_FIRST,
  p.S:categories[0]:sentiment::STRING AS COACH_TONE
FROM r JOIN p ON p.GAME_ID = r.GAME_ID AND p.TEAM_ID = r.TEAM_ID

### 2d. Football Ops: merge stats with text
`TEAM_STYLE_PROFILE` compares the style the **stats** show with the style the **media** describes and what the **coach claims**. `SAYS_VS_DOES_FLAG` is true when a coach claims to be run-first but the team passes on more than 55% of second-half plays.

In [ ]:
%%sql -r team_style_profile
-- Merge what the numbers say (stats) with what the words say (media + coach) for each team.
CREATE OR REPLACE TABLE SATURDAY_OPS.ANALYTICS.TEAM_STYLE_PROFILE AS
WITH st AS (
  SELECT TEAM_ID,
    SUM(PASS_ATT) / SUM(PASS_ATT + RUSH_ATT) AS PASS_RATE,
    SUM(SECOND_HALF_PASS_ATT) / SUM(SECOND_HALF_PASS_ATT + SECOND_HALF_RUSH_ATT) AS SECOND_HALF_PASS_RATE,
    AVG(PLAYS) AS PLAYS_PER_GAME,
    SUM(EXPLOSIVE_PLAYS) / SUM(PLAYS) AS EXPLOSIVE_RATE,
    AVG(FOURTH_DOWN_ATT) AS FOURTH_DOWN_ATT_PER_GAME
  FROM SATURDAY_OPS.RAW.TEAM_GAME_STATS GROUP BY TEAM_ID
), lbl AS (
  SELECT st.*,
    CASE WHEN PASS_RATE >= 0.60 THEN 'AIR_RAID'
         WHEN PLAYS_PER_GAME >= 80 THEN 'TEMPO_SPREAD'
         WHEN PASS_RATE <= 0.37 THEN 'GROUND_AND_POUND'
         WHEN PLAYS_PER_GAME <= 62 THEN 'DEFENSIVE_GRIND'
         ELSE 'BALANCED' END AS STAT_STYLE_LABEL
  FROM st
), txt AS (
  SELECT TEAM_ID, MODE(MEDIA_STYLE_LABEL) AS MEDIA_STYLE_LABEL,
    AVG(COACH_CLAIMS_RUN_FIRST::INT) AS RUN_FIRST_CLAIM_RATE
  FROM SATURDAY_OPS.ANALYTICS.GAME_TEXT_INSIGHTS GROUP BY TEAM_ID
), agree AS (
  SELECT g.TEAM_ID, AVG(IFF(g.MEDIA_STYLE_LABEL = l.STAT_STYLE_LABEL, 1, 0)) AS STYLE_AGREEMENT_PCT
  FROM SATURDAY_OPS.ANALYTICS.GAME_TEXT_INSIGHTS g JOIN lbl l USING (TEAM_ID) GROUP BY g.TEAM_ID
), idn AS (
  SELECT TEAM_ID, AI_SUMMARIZE_AGG(BODY) AS SEASON_IDENTITY
  FROM SATURDAY_OPS.RAW.GAME_RECAPS GROUP BY TEAM_ID
)
SELECT l.TEAM_ID, l.PASS_RATE, l.SECOND_HALF_PASS_RATE, l.PLAYS_PER_GAME, l.EXPLOSIVE_RATE,
  l.FOURTH_DOWN_ATT_PER_GAME, l.STAT_STYLE_LABEL, t.MEDIA_STYLE_LABEL, a.STYLE_AGREEMENT_PCT,
  t.RUN_FIRST_CLAIM_RATE,
  (t.RUN_FIRST_CLAIM_RATE >= 0.5 AND l.SECOND_HALF_PASS_RATE > 0.55) AS SAYS_VS_DOES_FLAG,
  i.SEASON_IDENTITY
FROM lbl l JOIN txt t USING (TEAM_ID) JOIN agree a USING (TEAM_ID) JOIN idn i USING (TEAM_ID)

In [ ]:
%%sql -r style_check
SELECT t.SCHOOL_NAME, p.STAT_STYLE_LABEL, p.MEDIA_STYLE_LABEL,
  ROUND(p.PASS_RATE * 100, 1) AS PASS_PCT,
  ROUND(p.SECOND_HALF_PASS_RATE * 100, 1) AS SECOND_HALF_PASS_PCT,
  ROUND(p.RUN_FIRST_CLAIM_RATE * 100) AS RUN_FIRST_CLAIM_PCT,
  p.SAYS_VS_DOES_FLAG, p.SEASON_IDENTITY
FROM SATURDAY_OPS.ANALYTICS.TEAM_STYLE_PROFILE p
JOIN SATURDAY_OPS.RAW.TEAMS t USING (TEAM_ID)
ORDER BY p.PASS_RATE DESC

In [ ]:
import matplotlib.pyplot as plt

# 12 rows, one per team, so collecting to pandas is safe.
df = style_check.to_pandas() if hasattr(style_check, "to_pandas") else style_check

fig, ax = plt.subplots(figsize=(10, 6))
for style, grp in df.groupby("STAT_STYLE_LABEL"):
    ax.scatter(grp["PASS_PCT"], grp["SECOND_HALF_PASS_PCT"], s=120, label=style)
for _, r in df.iterrows():
    ax.annotate(r["SCHOOL_NAME"] + ("  <- says run-first" if r["SAYS_VS_DOES_FLAG"] else ""),
                (r["PASS_PCT"], r["SECOND_HALF_PASS_PCT"]), textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.set_xlabel("Season pass rate (%)")
ax.set_ylabel("Second-half pass rate (%)")
ax.set_title("What the stats say: play style by team")
ax.legend(title="Stat style", fontsize=8)
plt.show()

### 2e. Game-day business rollup (plain SQL)

In [ ]:
%%sql -r game_business_summary
-- Plain SQL rollup of game-day revenue for each UCLA home game (Rose Bowl capacity 89,000).
CREATE OR REPLACE TABLE SATURDAY_OPS.ANALYTICS.GAME_BUSINESS_SUMMARY AS
WITH t AS (
  SELECT GAME_ID, COUNT(*) AS TICKETS_SOLD, SUM(SCANNED_FLAG::INT) AS ATTENDANCE, SUM(PRICE) AS TICKET_REVENUE
  FROM SATURDAY_OPS.RAW.TICKET_SALES GROUP BY GAME_ID
), c AS (
  SELECT GAME_ID, SUM(QTY * UNIT_PRICE) AS CONCESSION_REVENUE,
    SUM(IFF(GAME_QUARTER = 'Q4', QTY * UNIT_PRICE, 0)) AS Q4_CONCESSION_REVENUE
  FROM SATURDAY_OPS.RAW.CONCESSION_SALES GROUP BY GAME_ID
), m AS (
  SELECT GAME_ID, SUM(QTY * UNIT_PRICE) AS MERCH_REVENUE, SUM(QTY * (UNIT_PRICE - UNIT_COST)) AS MERCH_MARGIN
  FROM SATURDAY_OPS.RAW.MERCH_SALES GROUP BY GAME_ID
), d AS (
  SELECT g.GAME_ID, SUM(d.AMOUNT) AS DONATIONS_NEXT_7D
  FROM SATURDAY_OPS.RAW.GAMES g
  JOIN SATURDAY_OPS.RAW.DONATIONS d ON d.GIFT_DATE BETWEEN g.GAME_DATE AND DATEADD(day, 6, g.GAME_DATE)
  WHERE g.HOME_TEAM_ID = 1 GROUP BY g.GAME_ID
)
SELECT g.GAME_ID, g.SEASON_WEEK, g.GAME_DATE, g.KICKOFF_WINDOW, g.WEATHER, g.TEMP_F, g.IS_RIVALRY,
  o.SCHOOL_NAME AS OPPONENT, sp.STAT_STYLE_LABEL AS OPPONENT_STYLE, g.AWAY_RANK AS OPPONENT_RANK,
  g.HOME_SCORE AS UCLA_POINTS, g.AWAY_SCORE AS OPPONENT_POINTS,
  IFF(g.HOME_SCORE > g.AWAY_SCORE, 'W', 'L') AS RESULT,
  89000 AS CAPACITY, t.TICKETS_SOLD, t.ATTENDANCE, t.TICKET_REVENUE,
  c.CONCESSION_REVENUE, c.Q4_CONCESSION_REVENUE, m.MERCH_REVENUE, m.MERCH_MARGIN, d.DONATIONS_NEXT_7D
FROM SATURDAY_OPS.RAW.GAMES g
JOIN SATURDAY_OPS.RAW.TEAMS o ON o.TEAM_ID = g.AWAY_TEAM_ID
JOIN SATURDAY_OPS.ANALYTICS.TEAM_STYLE_PROFILE sp ON sp.TEAM_ID = g.AWAY_TEAM_ID
JOIN t ON t.GAME_ID = g.GAME_ID JOIN c ON c.GAME_ID = g.GAME_ID
JOIN m ON m.GAME_ID = g.GAME_ID JOIN d ON d.GAME_ID = g.GAME_ID
WHERE g.HOME_TEAM_ID = 1

In [ ]:
%%sql -r team_game_results
-- One row per team per game: stats, result, and what the media and coach said.
CREATE OR REPLACE VIEW SATURDAY_OPS.ANALYTICS.TEAM_GAME_RESULTS AS
SELECT s.GAME_ID, s.TEAM_ID, g.SEASON_WEEK, g.GAME_DATE, g.WEATHER,
  o.SCHOOL_NAME AS OPPONENT, s.IS_HOME,
  IFF(s.IS_HOME, g.HOME_SCORE, g.AWAY_SCORE) AS POINTS_FOR,
  IFF(s.IS_HOME, g.AWAY_SCORE, g.HOME_SCORE) AS POINTS_AGAINST,
  IFF(IFF(s.IS_HOME, g.HOME_SCORE, g.AWAY_SCORE) > IFF(s.IS_HOME, g.AWAY_SCORE, g.HOME_SCORE), 'W', 'L') AS RESULT,
  s.PLAYS, s.PASS_ATT, s.RUSH_ATT, s.PASS_YDS, s.RUSH_YDS, s.EXPLOSIVE_PLAYS, s.FOURTH_DOWN_ATT,
  s.FOURTH_DOWN_CONV, s.TURNOVERS, s.TIME_OF_POSSESSION_SEC, s.SECOND_HALF_PASS_ATT, s.SECOND_HALF_RUSH_ATT,
  i.MEDIA_STYLE_LABEL, i.KEY_PLAYER, i.TURNING_POINT, i.COACH_STATED_PLAN, i.COACH_CLAIMS_RUN_FIRST, i.COACH_TONE
FROM SATURDAY_OPS.RAW.TEAM_GAME_STATS s
JOIN SATURDAY_OPS.RAW.GAMES g ON g.GAME_ID = s.GAME_ID
JOIN SATURDAY_OPS.RAW.TEAMS o ON o.TEAM_ID = IFF(s.IS_HOME, g.AWAY_TEAM_ID, g.HOME_TEAM_ID)
JOIN SATURDAY_OPS.ANALYTICS.GAME_TEXT_INSIGHTS i ON i.GAME_ID = s.GAME_ID AND i.TEAM_ID = s.TEAM_ID

## 3. Semantic View: one source of truth for every business unit
The semantic view defines the business meaning once: tables, joins, metrics like **fill rate** and **post-game giving**, synonyms like "kit" or "big plays", custom instructions and verified queries. Cortex Analyst, Cortex Agents (Day 2) and BI tools all reuse it.

In [ ]:
%%sql -r semantic_view
CREATE OR REPLACE SEMANTIC VIEW SATURDAY_OPS.ANALYTICS.SATURDAY_OPS_SV
  TABLES (
    teams AS SATURDAY_OPS.RAW.TEAMS
      PRIMARY KEY (TEAM_ID)
      WITH SYNONYMS ('schools', 'programs')
      COMMENT = 'The 12 teams in the conference. UCLA is the home team.',
    team_style AS SATURDAY_OPS.ANALYTICS.TEAM_STYLE_PROFILE
      PRIMARY KEY (TEAM_ID)
      WITH SYNONYMS ('play style profile', 'team identity')
      COMMENT = 'Season play style per team: what the stats say vs what the media and coach say.',
    team_games AS SATURDAY_OPS.ANALYTICS.TEAM_GAME_RESULTS
      PRIMARY KEY (GAME_ID, TEAM_ID)
      WITH SYNONYMS ('box scores', 'game stats')
      COMMENT = 'One row per team per game with stats and AI-extracted recap and press conference insights.',
    home_games AS SATURDAY_OPS.ANALYTICS.GAME_BUSINESS_SUMMARY
      PRIMARY KEY (GAME_ID)
      WITH SYNONYMS ('game day', 'home game business')
      COMMENT = 'Game-day business results for each UCLA home game at the Rose Bowl.',
    fans AS SATURDAY_OPS.RAW.FANS
      PRIMARY KEY (FAN_ID)
      COMMENT = 'UCLA fan base: season ticket holders, single-game buyers, alumni, students.',
    tickets AS SATURDAY_OPS.RAW.TICKET_SALES
      PRIMARY KEY (TICKET_ID)
      COMMENT = 'One row per ticket sold for UCLA home games.',
    fan_feedback AS SATURDAY_OPS.ANALYTICS.FAN_INSIGHTS
      PRIMARY KEY (SURVEY_ID)
      WITH SYNONYMS ('fan surveys', 'survey responses')
      COMMENT = 'Post-game fan surveys scored with AI_SENTIMENT.',
    fan_cases AS SATURDAY_OPS.ANALYTICS.TICKET_INSIGHTS
      PRIMARY KEY (CASE_ID)
      WITH SYNONYMS ('fan service tickets', 'complaints', 'support cases')
      COMMENT = 'Fan service tickets classified with AI_CLASSIFY and AI_EXTRACT.'
  )
  RELATIONSHIPS (
    team_style_to_teams AS team_style (TEAM_ID) REFERENCES teams,
    team_games_to_teams AS team_games (TEAM_ID) REFERENCES teams,
    tickets_to_home_games AS tickets (GAME_ID) REFERENCES home_games,
    tickets_to_fans AS tickets (FAN_ID) REFERENCES fans,
    feedback_to_home_games AS fan_feedback (GAME_ID) REFERENCES home_games,
    feedback_to_fans AS fan_feedback (FAN_ID) REFERENCES fans,
    cases_to_home_games AS fan_cases (GAME_ID) REFERENCES home_games
  )
  FACTS (
    team_games.plays AS PLAYS,
    team_games.pass_attempts AS PASS_ATT,
    team_games.rush_attempts AS RUSH_ATT,
    team_games.second_half_pass_attempts AS SECOND_HALF_PASS_ATT,
    team_games.second_half_rush_attempts AS SECOND_HALF_RUSH_ATT,
    team_games.explosive_plays AS EXPLOSIVE_PLAYS COMMENT = 'Plays of 20 or more yards',
    team_games.fourth_down_attempts AS FOURTH_DOWN_ATT,
    team_games.turnovers AS TURNOVERS,
    team_games.points_for AS POINTS_FOR,
    team_games.points_against AS POINTS_AGAINST,
    home_games.capacity AS CAPACITY,
    home_games.tickets_sold AS TICKETS_SOLD,
    home_games.attendance AS ATTENDANCE COMMENT = 'Tickets scanned at the gate',
    home_games.ticket_revenue AS TICKET_REVENUE,
    home_games.concession_revenue AS CONCESSION_REVENUE,
    home_games.q4_concession_revenue AS Q4_CONCESSION_REVENUE,
    home_games.merch_revenue AS MERCH_REVENUE,
    home_games.merch_margin AS MERCH_MARGIN,
    home_games.donations_next_7d AS DONATIONS_NEXT_7D COMMENT = 'Donations in the 7 days starting on game day',
    tickets.ticket_price AS PRICE,
    fan_feedback.sentiment_score AS SENTIMENT_SCORE COMMENT = '1 positive, 0 neutral or mixed, -1 negative',
    fan_feedback.survey_rating AS OVERALL_RATING,
    team_style.run_first_claim_rate AS RUN_FIRST_CLAIM_RATE COMMENT = 'Share of press conferences where the coach claims to be run-first',
    team_style.season_second_half_pass_rate AS SECOND_HALF_PASS_RATE COMMENT = 'Season second-half pass rate',
    team_style.season_pass_rate AS PASS_RATE COMMENT = 'Season pass rate'
  )
  DIMENSIONS (
    teams.team_id AS TEAM_ID,
    team_style.team_id AS TEAM_ID,
    team_games.game_id AS GAME_ID,
    team_games.team_id AS TEAM_ID,
    home_games.game_id AS GAME_ID,
    fans.fan_id AS FAN_ID,
    tickets.ticket_id AS TICKET_ID,
    tickets.game_id AS GAME_ID,
    tickets.fan_id AS FAN_ID,
    fan_feedback.survey_id AS SURVEY_ID,
    fan_feedback.game_id AS GAME_ID,
    fan_feedback.fan_id AS FAN_ID,
    fan_cases.case_id AS CASE_ID,
    fan_cases.game_id AS GAME_ID,
    teams.school_name AS SCHOOL_NAME WITH SYNONYMS ('team', 'school', 'program') COMMENT = 'Team name, for example UCLA or BYU',
    teams.mascot AS MASCOT,
    teams.coach AS COACH_ALIAS WITH SYNONYMS ('head coach'),
    teams.stadium AS STADIUM,
    team_style.stat_style_label AS STAT_STYLE_LABEL
      WITH SYNONYMS ('play style', 'offensive style', 'offense type', 'style')
      COMMENT = 'Play style derived from stats: AIR_RAID, TEMPO_SPREAD, GROUND_AND_POUND, DEFENSIVE_GRIND, BALANCED',
    team_style.media_style_label AS MEDIA_STYLE_LABEL
      WITH SYNONYMS ('media style') COMMENT = 'Play style the media recaps describe, classified by AI',
    team_style.says_vs_does_flag AS SAYS_VS_DOES_FLAG
      WITH SYNONYMS ('says vs does', 'claims run first but passes')
      COMMENT = 'TRUE when the coach claims run-first but the team passes on more than 55% of second-half plays',
    team_style.season_identity AS SEASON_IDENTITY COMMENT = 'AI summary of the season from game recaps',
    team_games.season_week AS SEASON_WEEK WITH SYNONYMS ('week'),
    team_games.game_date AS GAME_DATE,
    team_games.opponent AS OPPONENT,
    team_games.result AS RESULT WITH SYNONYMS ('win or loss', 'outcome') COMMENT = 'W or L from this team perspective',
    team_games.is_home AS IS_HOME,
    team_games.weather AS WEATHER,
    team_games.game_media_style AS MEDIA_STYLE_LABEL COMMENT = 'AI play style label for this single game recap',
    team_games.coach_stated_plan AS COACH_STATED_PLAN WITH SYNONYMS ('game plan'),
    team_games.coach_claims_run_first AS COACH_CLAIMS_RUN_FIRST,
    team_games.coach_tone AS COACH_TONE,
    team_games.key_player AS KEY_PLAYER,
    team_games.turning_point AS TURNING_POINT,
    home_games.home_week AS SEASON_WEEK WITH SYNONYMS ('home game week'),
    home_games.home_game_date AS GAME_DATE,
    home_games.home_opponent AS OPPONENT WITH SYNONYMS ('opponent', 'visiting team'),
    home_games.opponent_style AS OPPONENT_STYLE WITH SYNONYMS ('opponent play style', 'opponent offense', 'air raid opponents') COMMENT = 'Play style of the visiting opponent. Use this for attendance or revenue by opponent style.',
    home_games.opponent_rank AS OPPONENT_RANK,
    home_games.kickoff_window AS KICKOFF_WINDOW WITH SYNONYMS ('kickoff time') COMMENT = 'NOON, AFTERNOON or NIGHT',
    home_games.home_weather AS WEATHER COMMENT = 'CLEAR, RAIN, HEAT or COLD',
    home_games.is_rivalry AS IS_RIVALRY WITH SYNONYMS ('rivalry game'),
    home_games.home_result AS RESULT WITH SYNONYMS ('UCLA result') COMMENT = 'W or L for UCLA',
    fans.fan_segment AS FAN_SEGMENT WITH SYNONYMS ('fan type', 'season ticket holder')
      COMMENT = 'SEASON_TICKET, SINGLE_GAME, ALUMNI or STUDENT',
    fans.donor_tier AS DONOR_TIER COMMENT = 'NONE, BRONZE, SILVER or GOLD',
    fans.age_band AS AGE_BAND,
    tickets.seat_tier AS SEAT_TIER COMMENT = 'LOWER, UPPER, PREMIUM or STUDENT',
    tickets.purchase_channel AS PURCHASE_CHANNEL,
    tickets.section AS SECTION,
    fan_feedback.overall_sentiment AS OVERALL_SENTIMENT,
    fan_feedback.food_sentiment AS FOOD_SENTIMENT,
    fan_feedback.parking_sentiment AS PARKING_SENTIMENT,
    fan_feedback.seats_sentiment AS SEATS_SENTIMENT,
    fan_feedback.team_sentiment AS TEAM_SENTIMENT,
    fan_feedback.price_sentiment AS PRICE_SENTIMENT,
    fan_feedback.survey_comment AS COMMENT,
    fan_cases.complaint_category AS CATEGORY
      WITH SYNONYMS ('complaint type', 'issue category')
      COMMENT = 'PARKING, SEATING, REFUND, ACCESSIBILITY, CONCESSIONS, TICKETING_APP or SECURITY',
    fan_cases.issue_summary AS ISSUE_SUMMARY,
    fan_cases.gate AS GATE,
    fan_cases.case_status AS STATUS
  )
  METRICS (
    team_games.games_played AS COUNT(team_games.GAME_ID),
    team_games.win_pct AS AVG(IFF(team_games.RESULT = 'W', 1, 0)) WITH SYNONYMS ('win percentage', 'win rate'),
    team_games.pass_rate AS SUM(team_games.pass_attempts) / NULLIF(SUM(team_games.pass_attempts + team_games.rush_attempts), 0)
      WITH SYNONYMS ('pass percentage') COMMENT = 'Share of offensive plays that were passes',
    team_games.second_half_pass_rate AS SUM(team_games.second_half_pass_attempts)
      / NULLIF(SUM(team_games.second_half_pass_attempts + team_games.second_half_rush_attempts), 0),
    team_games.explosive_play_rate AS SUM(team_games.explosive_plays) / NULLIF(SUM(team_games.plays), 0) WITH SYNONYMS ('big play rate'),
    team_games.plays_per_game AS AVG(team_games.plays) WITH SYNONYMS ('tempo', 'pace'),
    team_games.fourth_down_attempts_per_game AS AVG(team_games.fourth_down_attempts) WITH SYNONYMS ('fourth down aggressiveness'),
    team_games.avg_points_scored AS AVG(team_games.points_for),
    home_games.total_revenue AS SUM(home_games.ticket_revenue + home_games.concession_revenue + home_games.merch_revenue)
      WITH SYNONYMS ('game day revenue'),
    home_games.ticket_revenue_total AS SUM(home_games.ticket_revenue),
    home_games.concession_revenue_total AS SUM(home_games.concession_revenue),
    home_games.merch_revenue_total AS SUM(home_games.merch_revenue),
    home_games.total_attendance AS SUM(home_games.attendance),
    -- LIVE STEP: uncomment the next line and re-run this cell to add a new metric
    -- home_games.per_cap_spend AS SUM(home_games.concession_revenue + home_games.merch_revenue) / NULLIF(SUM(home_games.attendance), 0) WITH SYNONYMS ('per cap', 'spend per fan') COMMENT = 'Concession plus merch revenue per attendee',
    home_games.attendance_pct AS SUM(home_games.attendance) / NULLIF(SUM(home_games.capacity), 0)
      WITH SYNONYMS ('fill rate', 'capacity utilization') COMMENT = 'Scanned attendance divided by Rose Bowl capacity',
    home_games.no_show_rate AS 1 - SUM(home_games.attendance) / NULLIF(SUM(home_games.tickets_sold), 0),
    home_games.q4_concession_share AS SUM(home_games.q4_concession_revenue) / NULLIF(SUM(home_games.concession_revenue), 0)
      COMMENT = 'Share of concession revenue sold in the 4th quarter',
    home_games.donations_after_game AS SUM(home_games.donations_next_7d) WITH SYNONYMS ('post-game giving', 'donations'),
    tickets.avg_ticket_price AS AVG(tickets.ticket_price),
    fan_feedback.avg_fan_sentiment AS AVG(fan_feedback.sentiment_score) WITH SYNONYMS ('fan mood'),
    fan_feedback.avg_survey_rating AS AVG(fan_feedback.survey_rating),
    fan_feedback.survey_count AS COUNT(fan_feedback.SURVEY_ID),
    fan_cases.complaint_count AS COUNT(fan_cases.CASE_ID) WITH SYNONYMS ('number of complaints', 'ticket volume')
  )
  COMMENT = 'Saturday Ops: UCLA athletic department game-day business plus conference-wide play style. Synthetic data.'
  AI_SQL_GENERATION 'UCLA is the home team; we, our and us mean UCLA. Business data (tickets, attendance, revenue, concessions, merch, donations, fan surveys, fan service tickets) exists only for UCLA home games at the Rose Bowl in the 2025 season. Football stats and play styles exist for all 12 teams. For questions that combine home game business metrics (attendance, revenue, donations, complaints, sentiment) with opponent play style, use home_games.opponent_style directly and do not join team_style. Show percentages rounded to 1 decimal place.'
  AI_VERIFIED_QUERIES (
    attendance_by_home_game AS (
      QUESTION 'What was the attendance percentage and no-show rate for each home game?'
      ONBOARDING_QUESTION TRUE
      SQL 'SELECT home_week, home_opponent, ROUND(SUM(attendance) / SUM(capacity) * 100, 1) AS attendance_pct, ROUND((1 - SUM(attendance) / SUM(tickets_sold)) * 100, 1) AS no_show_rate FROM home_games GROUP BY home_week, home_opponent ORDER BY home_week'
    ),
    revenue_by_kickoff AS (
      QUESTION 'Total game day revenue by kickoff window'
      SQL 'SELECT kickoff_window, SUM(ticket_revenue + concession_revenue + merch_revenue) AS total_revenue FROM home_games GROUP BY kickoff_window ORDER BY total_revenue DESC'
    ),
    rivalry_vs_other_revenue AS (
      QUESTION 'How does revenue per attendee in the USC rivalry game compare to other home games?'
      SQL 'SELECT is_rivalry, SUM(ticket_revenue + concession_revenue + merch_revenue) / SUM(attendance) AS revenue_per_attendee FROM home_games GROUP BY is_rivalry'
    ),
    giving_after_wins_losses AS (
      QUESTION 'Do donations go up after wins compared to losses?'
      ONBOARDING_QUESTION TRUE
      SQL 'SELECT home_result, AVG(donations_next_7d) AS avg_donations_next_7_days FROM home_games GROUP BY home_result'
    ),
    team_play_styles AS (
      QUESTION 'What is each team play style and pass rate?'
      ONBOARDING_QUESTION TRUE
      SQL 'SELECT t.school_name, s.stat_style_label, ROUND(SUM(g.pass_attempts) / SUM(g.pass_attempts + g.rush_attempts) * 100, 1) AS pass_rate_pct FROM team_games AS g JOIN teams AS t ON g.team_id = t.team_id JOIN team_style AS s ON s.team_id = t.team_id GROUP BY t.school_name, s.stat_style_label ORDER BY pass_rate_pct DESC'
    ),
    ucla_win_pct_by_opponent_style AS (
      QUESTION 'What is UCLA win percentage against each opponent play style?'
      SQL 'SELECT os.stat_style_label AS opponent_style, AVG(IFF(g.result = ''W'', 1, 0)) AS win_pct FROM team_games AS g JOIN teams AS t ON g.team_id = t.team_id JOIN teams AS o ON o.school_name = g.opponent JOIN team_style AS os ON os.team_id = o.team_id WHERE t.school_name = ''UCLA'' GROUP BY os.stat_style_label'
    ),
    says_vs_does AS (
      QUESTION 'Which coaches say they are run-first but actually pass a lot?'
      ONBOARDING_QUESTION TRUE
      SQL 'SELECT t.school_name, t.coach, s.run_first_claim_rate, s.season_second_half_pass_rate FROM team_style AS s JOIN teams AS t ON s.team_id = t.team_id WHERE s.says_vs_does_flag = TRUE'
    ),
    attendance_vs_air_raid AS (
      QUESTION 'Does attendance go up against air raid teams?'
      SQL 'SELECT IFF(opponent_style = ''AIR_RAID'', ''AIR_RAID'', ''OTHER'') AS opponent_group, SUM(attendance) / SUM(capacity) AS attendance_pct FROM home_games GROUP BY opponent_group'
    ),
    sentiment_by_week AS (
      QUESTION 'Average fan sentiment by home game week'
      SQL 'SELECT h.home_week, h.home_opponent, AVG(f.sentiment_score) AS avg_fan_sentiment FROM fan_feedback AS f JOIN home_games AS h ON f.game_id = h.game_id GROUP BY h.home_week, h.home_opponent ORDER BY h.home_week'
    ),
    miami_complaints AS (
      QUESTION 'What were the top complaint categories at the Miami game?'
      SQL 'SELECT c.complaint_category, COUNT(c.case_id) AS complaint_count FROM fan_cases AS c JOIN home_games AS h ON c.game_id = h.game_id WHERE h.home_opponent = ''Miami'' GROUP BY c.complaint_category ORDER BY complaint_count DESC LIMIT 3'
    )
  )

**Live step:** in the cell above, uncomment the `per_cap_spend` line and run the cell again. You just added a governed business metric that every consumer of the semantic view now shares.

Query the semantic view directly with SQL:

In [ ]:
%%sql -r query_semantic_view
SELECT * FROM SEMANTIC_VIEW(
  SATURDAY_OPS.ANALYTICS.SATURDAY_OPS_SV
  DIMENSIONS home_games.home_week, home_games.home_opponent, home_games.opponent_style, home_games.home_result
  METRICS home_games.attendance_pct, home_games.total_revenue, home_games.donations_after_game
)
ORDER BY home_week

### Ask it in plain English
In Snowsight, open **AI & ML > Cortex Analyst**, select `SATURDAY_OPS.ANALYTICS.SATURDAY_OPS_SV`, and try:
- *Did the Rose Bowl fill better against air-raid teams?*
- *Which coaches say they're run-first but aren't?*
- *What were the top complaint categories at the Miami game?*
- *What was our per cap spend at the USC game?* (after the live step)

## 4. Cortex Search: find answers inside the text
Not every question is a number. Cortex Search gives hybrid (keyword plus semantic) search over coach press conferences, with filters for team, week, result and the AI-derived media style. The service takes about 90 seconds to build.

In [ ]:
%%sql -r create_search
-- One Cortex Search service over coach press conference transcripts,
-- with filters for team, week, result and the AI-derived media style.
CREATE OR REPLACE CORTEX SEARCH SERVICE SATURDAY_OPS.ANALYTICS.COACH_PRESSER_SEARCH
  ON TRANSCRIPT
  ATTRIBUTES SCHOOL_NAME, SEASON_WEEK, RESULT, MEDIA_STYLE_LABEL
  WAREHOUSE = SATURDAY_OPS_WH
  TARGET_LAG = '1 day'
  AS
  SELECT p.PRESSER_ID, p.TRANSCRIPT, p.COACH_ALIAS, t.SCHOOL_NAME, g.SEASON_WEEK,
         r.OPPONENT, r.RESULT, r.MEDIA_STYLE_LABEL
  FROM SATURDAY_OPS.RAW.PRESSER_TRANSCRIPTS p
  JOIN SATURDAY_OPS.RAW.TEAMS t ON t.TEAM_ID = p.TEAM_ID
  JOIN SATURDAY_OPS.RAW.GAMES g ON g.GAME_ID = p.GAME_ID
  JOIN SATURDAY_OPS.ANALYTICS.TEAM_GAME_RESULTS r ON r.GAME_ID = p.GAME_ID AND r.TEAM_ID = p.TEAM_ID

In [ ]:
%%sql -r search_run_game
SELECT r.value:SCHOOL_NAME::STRING AS SCHOOL, r.value:SEASON_WEEK::INT AS WEEK,
       r.value:RESULT::STRING AS RESULT, r.value:TRANSCRIPT::STRING AS TRANSCRIPT
FROM TABLE(FLATTEN(PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'SATURDAY_OPS.ANALYTICS.COACH_PRESSER_SEARCH',
  '{"query": "coach talked about establishing the run after a loss",
    "columns": ["SCHOOL_NAME", "SEASON_WEEK", "RESULT", "TRANSCRIPT"],
    "filter": {"@eq": {"RESULT": "L"}},
    "limit": 5}'
)):results)) r

In [ ]:
%%sql -r search_fourth_down
SELECT r.value:SCHOOL_NAME::STRING AS SCHOOL, r.value:SEASON_WEEK::INT AS WEEK,
       r.value:RESULT::STRING AS RESULT, r.value:TRANSCRIPT::STRING AS TRANSCRIPT
FROM TABLE(FLATTEN(PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'SATURDAY_OPS.ANALYTICS.COACH_PRESSER_SEARCH',
  '{"query": "going for it on fourth down",
    "columns": ["SCHOOL_NAME", "SEASON_WEEK", "RESULT", "TRANSCRIPT"],
    "filter": {"@eq": {"SCHOOL_NAME": "BYU"}},
    "limit": 5}'
)):results)) r

## 5. The payoff: structured plus unstructured, side by side
One view of every UCLA home game: the business result, the fan voice, and what the coach said versus what the team did. Notice the coach describes a run-first identity every week while the Bruins pass on roughly 6 of every 10 second-half snaps.

In [ ]:
%%sql -r payoff
WITH sentiment AS (
  SELECT GAME_ID, AVG(SENTIMENT_SCORE) AS FAN_SENTIMENT FROM SATURDAY_OPS.ANALYTICS.FAN_INSIGHTS GROUP BY GAME_ID
), complaints AS (
  SELECT GAME_ID, COUNT(*) AS COMPLAINTS, MODE(CATEGORY) AS TOP_COMPLAINT
  FROM SATURDAY_OPS.ANALYTICS.TICKET_INSIGHTS GROUP BY GAME_ID
)
SELECT h.SEASON_WEEK AS WEEK, h.OPPONENT, h.OPPONENT_STYLE, h.RESULT, h.WEATHER,
  ROUND(h.ATTENDANCE / h.CAPACITY * 100, 1) AS FILL_PCT,
  ROUND((h.CONCESSION_REVENUE + h.MERCH_REVENUE) / h.ATTENDANCE, 2) AS PER_CAP_SPEND,
  ROUND(h.DONATIONS_NEXT_7D) AS GIVING_NEXT_7D,
  ROUND(s.FAN_SENTIMENT, 2) AS FAN_SENTIMENT, c.COMPLAINTS, c.TOP_COMPLAINT,
  r.COACH_STATED_PLAN AS UCLA_COACH_SAYS,
  ROUND(r.SECOND_HALF_PASS_ATT / NULLIF(r.SECOND_HALF_PASS_ATT + r.SECOND_HALF_RUSH_ATT, 0) * 100, 1) AS UCLA_2H_PASS_PCT
FROM SATURDAY_OPS.ANALYTICS.GAME_BUSINESS_SUMMARY h
JOIN sentiment s ON s.GAME_ID = h.GAME_ID
JOIN complaints c ON c.GAME_ID = h.GAME_ID
JOIN SATURDAY_OPS.ANALYTICS.TEAM_GAME_RESULTS r ON r.GAME_ID = h.GAME_ID AND r.TEAM_ID = 1
ORDER BY WEEK

### What you built today
- 13 raw tables generated in one SQL cell
- 4 AI-enriched tables and 1 merged play style profile, all built with SQL and AI Functions
- 1 semantic view (`SATURDAY_OPS_SV`) with business metrics, synonyms, custom instructions and 10 verified queries
- 1 Cortex Search service (`COACH_PRESSER_SEARCH`)

**Day 2:** put a Cortex Agent on top of the semantic view and the search service, deliver it to each business unit in CoWork, and keep the semantic layer as the source of truth for AI and BI dashboards.

## 6. Cleanup (optional)
Keep everything for Day 2. When you are done with the bootcamp, uncomment and run the cell below.

In [ ]:
# session.sql("DROP DATABASE IF EXISTS SATURDAY_OPS").collect()
# session.sql("DROP WAREHOUSE IF EXISTS SATURDAY_OPS_WH").collect()